# Ricostruzione tomografica in TD-DOT: dal modello diretto alle regolarizzazioni L1 e L1+TV

Questo notebook spiega, blocco per blocco, lo script `scan_ricostruzioni.py`. Lo script simula misure di **tomografia ottica diffusa nel dominio del tempo** (TD-DOT) in riflettanza e le ricostruisce con quattro famiglie di metodi: TSVD, Tikhonov, L1 e L1 + variazione totale (L1+TV). Il confronto è ripetuto al variare di lunghezza d'onda, posizione dell'inclusione e variazione di composizione.

Per ogni blocco di codice ci sono due parti:

- **teoria**: cosa calcola il blocco, con le formule scritte nelle stesse variabili del codice;
- **implementazione**: come la teoria diventa codice Python (strutture dati, `numpy`, scelte numeriche).

Le sezioni sulle regolarizzazioni **L1** e **L1+TV** sono le più dettagliate: ripercorrono tutti i passaggi algebrici, dall'idea del metodo fino all'iterazione implementata.

**Materiale di riferimento**

- S. Okawa, Y. Hoshi, *A Review of Image Reconstruction Algorithms for Diffuse Optical Tomography*, Appl. Sci. **13**, 5016 (2023). Nel testo: "la rassegna", con i numeri di paragrafo ed equazione.
- Appunti *Regolarizzazione L1 (FISTA) senza vincolo di segno*: soglia morbida, condizioni di ottimalità, FISTA, scelta di $\lambda$.

**Come usare il notebook**

1. Impostare `file_spettri` (cella dello STEP 0) sul percorso del file Excel con gli spettri specifici.
2. Eseguire le celle in ordine: il notebook *è* lo script, diviso in blocchi.
3. Lo scan completo (31 lunghezze d'onda × 3 posizioni × 6 variazioni, 6 metodi) richiede circa un'ora e mezza. Per una prova veloce si possono ridurre `wl_nm`, `posizioni_scan`, `variazioni_scan`, `alpha_tv_lista` e `n_realizzazioni_rumore`.

**Sistema di unità**: lunghezze in mm, tempi in ns, coefficienti ottici in mm⁻¹, velocità in mm/ns ($c = 300$ mm/ns), concentrazioni in µM.

## Indice

0. [Quadro generale e notazione](#quadro)
1. [Spettri di assorbimento e scattering (STEP 0)](#step0)
2. [Geometria, griglia di voxel e gate temporali (STEP 1)](#step1)
3. [Inclusioni e variazioni di composizione (STEP 2)](#step2)
4. [Modello diretto: diffusione, perturbazione di Born, matrice W](#forward)
5. [Modello del rumore e sbiancamento delle misure](#rumore)
6. [TSVD e Tikhonov](#tikhonov)
7. [Regolarizzazione L1 con FISTA](#l1)
8. [Regolarizzazione L1 + variazione totale con ADMM](#l1tv)
9. [Soglie di significatività, componenti connesse e metriche](#metriche)
10. [Salvataggio in Excel](#excel)
11. [Preparazione per lunghezza d'onda e calibrazione sul rumore](#prep)
12. [Ciclo principale dello scan](#ciclo)
13. [Riepilogo](#riepilogo)
14. [Considerazioni finali: verso una ricostruzione più realistica](#finali)
15. [Riferimenti](#riferimenti)

<a id="quadro"></a>
## 0. Quadro generale e notazione

La DOT ricostruisce mappe 3D delle proprietà ottiche ($\mu_a$, $\mu_s'$) illuminando e rivelando luce nel vicino infrarosso solo dalla superficie. A differenza della TC a raggi X, i fotoni non viaggiano in linea retta ma diffondono: ogni misura "vede" un volume a forma di banana fra sorgente e rivelatore. La ricostruzione si divide quindi in due processi (rassegna, §1–2):

1. **processo diretto** (forward): date le proprietà ottiche, prevedere le misure. Qui si usa l'approssimazione di diffusione in un mezzo semi-infinito e la perturbazione di Born al primo ordine, che porta a un modello **lineare**
$$\mathbf M \approx W\,\mathbf A ;$$
2. **processo inverso** (ottimizzazione): dalle misure risalire a $\mathbf A=\delta\boldsymbol\mu_a$, cioè alla variazione di assorbimento voxel per voxel. Il problema è mal posto: le incognite sono più delle misure informative, e il rumore viene amplificato. Serve una **regolarizzazione**, cioè un'informazione a priori sull'oggetto.

Tutti i metodi dello script risolvono un problema della forma

$$
\hat{\mathbf A}=\arg\min_{\mathbf A}\;\underbrace{\tfrac12\big\|W_w\mathbf A-\mathbf m_w\big\|_2^2}_{\text{fedeltà ai dati }(\chi^2/2)}\;+\;\underbrace{\lambda\,R(\mathbf A)}_{\text{regolarizzazione}}
$$

e si distinguono per la scelta di $R$:

| Metodo | $R(\mathbf A)$ | Cosa favorisce |
|---|---|---|
| Tikhonov | $\lVert\mathbf A\rVert_2^2$ | valori piccoli e diffusi |
| TSVD | (troncamento dei modi della SVD) | solo le componenti ben misurate |
| L1 | $\sum_j n_j\lvert A_j\rvert$ | pochi voxel accesi (sparsità) |
| L1+TV | $\alpha\sum_j p_j\lvert A_j\rvert+(1-\alpha)\sum_{(i,k)}q_{ik}\lvert A_i-A_k\rvert$ | poche regioni compatte e uniformi |

### Notazione

| Simbolo | Significato | Variabile nel codice |
|---|---|---|
| $s,\ d$ | sorgente, rivelatore | `s`, `d` |
| $p$ | coppia sorgente-rivelatore | `p`, lista `pairs` |
| $g$ | gate temporale | `g`, lista `gates` |
| $m=p\,N_\text{gates}+g$ | indice della misura | riga di `W` |
| $j$ | voxel | colonna di `W`, `N_vox` = 2048 |
| $\mathbf A$ | $\delta\mu_a$ per voxel [mm⁻¹] | `A` (vero), `A_hat` (ricostruito) |
| $W$ | matrice di sensibilità (Jacobiano) | `W` |
| $M_m$ | misura: variazione relativa dei conteggi | `M_ideal`, `M_noisy` |
| $\sigma_m$ | deviazione standard di $M_m$ | `sig0`, `sig_hat` |
| $W_w,\ \mathbf m_w$ | matrice e misure pesate ($/\sigma_m$) | `W_w`, `m_w` |
| $n_j=\lVert\mathbf w_j\rVert$ | norma della colonna $j$ di $W_w$ | `col_L1` |
| $U,\ s_k,\ V$ | SVD di $W_w$ | `Uw`, `sw`, `Vtw` |
| $c_k=\mathbf u_k^T\mathbf m_w$ | coefficienti delle misure sui modi | `c` |

### Librerie

- `numpy`: tutto il calcolo vettoriale. `from numpy import pi, exp` serve solo a scrivere le formule in modo compatto.
- `pandas`: lettura degli spettri da Excel e scrittura dei risultati.
- `scipy.sparse`: la matrice delle differenze finite della variazione totale, che ha due elementi non nulli per riga.
- `time`: misura dei tempi e stima del tempo rimanente.

In [ ]:
import time
import numpy as np
import pandas as pd
from numpy import pi, exp
from scipy import sparse

<a id="step0"></a>
## 1. Spettri di assorbimento e scattering (STEP 0)

### Teoria

Nel vicino infrarosso l'assorbimento dei tessuti è dovuto a pochi cromofori. Il coefficiente di assorbimento del background è la loro somma pesata (rassegna, eq. 29):

$$
\mu_a(\lambda)=0.1\Big[C_\text{HHb}\,\varepsilon_\text{HHb}(\lambda)+C_\mathrm{O_2Hb}\,\varepsilon_\mathrm{O_2Hb}(\lambda)+f_\text{lip}\,\mu_{a,\text{lip}}(\lambda)+f_w\,\mu_{a,w}(\lambda)\Big]
$$

- $C$ in µM ed $\varepsilon$ in cm⁻¹/µM per le emoglobine;
- $f$ frazione in volume e $\mu_a$ del componente puro (cm⁻¹) per lipidi e acqua;
- il fattore 0.1 converte da cm⁻¹ a mm⁻¹.

Lo **scattering ridotto** segue la legge di potenza empirica (Jacques, Phys. Med. Biol. 58, R37, 2013, tessuto adiposo):

$$
\mu_s'(\lambda)=a\left(\frac{\lambda}{\lambda_0}\right)^{-b},\qquad a=1.84\ \text{mm}^{-1},\ b=0.672,\ \lambda_0=500\ \text{nm}.
$$

### Implementazione

- `composizione` è un dizionario `{cromoforo: peso}`; lo stesso formato serve per le variazioni dell'inclusione.
- `np.arange(650, 951, 10)` esclude l'estremo superiore, quindi si scrive 951 per includere 950 nm: 31 lunghezze d'onda.
- `a_jac = 18.4 / 10` converte il valore tabulato in cm⁻¹ in mm⁻¹.

In [ ]:
# %% =============================================================
# STEP 0 - Spettri
# ==================================================================
file_spettri = r"C:\Users\marti\Downloads\specific_absorption.xlsx"
foglio_spettri = "new"                     # NON usare "mod": colonna Lipid disallineata
file_risultati = "risultati_scan.xlsx"     # file Excel di output (riscritto a ogni lambda)

composizione = {
    "HHb":   50.0,    # [uM]  emoglobina deossigenata
    "O2Hb": 150.0,    # [uM]  emoglobina ossigenata
    "Lipid":  0.70,   # [-]   frazione in volume di lipide
    "H2O":    0.30,   # [-]   frazione in volume di acqua
}

wl_nm = np.arange(650, 951, 10, dtype=float)   # [nm] 650-950 nm, passo 10, estremi inclusi
N_wl = wl_nm.size

# scattering ridotto: Jacques, Phys. Med. Biol. 58 R37 (2013), Tab. 2, "fatty tissue"
a_jac = 18.4 / 10     # [mm^-1] mu_s' a lambda0 (18.4 cm^-1 -> 1.84 mm^-1)
b_jac = 0.672         # [-]     scattering power
wl0_jac = 500.0       # [nm]    lunghezza d'onda di riferimento

### Funzioni sugli spettri

- `leggi_spettri`: `pd.read_excel(..., skiprows=4)` salta le quattro righe di intestazione. `pd.to_numeric(..., errors="coerce")` trasforma in `NaN` le righe non numeriche, che `.notna()` scarta. La *dict comprehension* finale restituisce un array per colonna.
- `mua_background`: `np.interp(wl, L, spettro)` interpola linearmente lo spettro tabulato sulle lunghezze d'onda richieste. Il ciclo `for nome, peso in comp.items()` realizza la somma $\sum_c C_c\,\varepsilon_c(\lambda)$; il controllo iniziale evita di estrapolare fuori dal range del file.
- `mus_jacques`: la legge di potenza, applicata a tutto il vettore `wl` in un colpo solo (operazioni elemento per elemento di `numpy`).

In [ ]:
def leggi_spettri(percorso, foglio="new"):
    """Spettri specifici dei cromofori dal file Excel (righe 1-4 = intestazioni)."""
    df = pd.read_excel(percorso, sheet_name=foglio, skiprows=4)
    df = df[pd.to_numeric(df["Lambda"], errors="coerce").notna()]
    return {k: df[k].to_numpy(dtype=float) for k in ("Lambda", "HHb", "O2Hb", "Lipid", "H2O")}


def mua_background(spettri, comp, wl):
    """mu_a [mm^-1] = 0.1 * [C_HHb eps_HHb + C_O2Hb eps_O2Hb + f_lip mua_lip + f_w mua_w]"""
    L = spettri["Lambda"]
    if wl.min() < L.min() or wl.max() > L.max():
        raise ValueError(f"Lunghezze d'onda {wl.min():.0f}-{wl.max():.0f} nm fuori dal "
                         f"range del file ({L.min():.0f}-{L.max():.0f} nm).")
    mua_cm = np.zeros_like(wl)
    for nome, peso in comp.items():
        mua_cm += peso * np.interp(wl, L, spettri[nome])
    return mua_cm / 10


def mus_jacques(wl, a=a_jac, b=b_jac, wl0=wl0_jac):
    """mu_s' [mm^-1] dalla legge di potenza di Jacques."""
    return a * (wl / wl0)**(-b)

### Variazioni di composizione dell'inclusione

L'inclusione differisce dal background per una variazione di composizione $\delta C$. Poiché $\mu_a$ è **lineare** nelle concentrazioni, la variazione di assorbimento si ottiene applicando la stessa formula alle sole variazioni:

$$
\delta\mu_a(\lambda)=0.1\Big[\delta C_\text{HHb}\,\varepsilon_\text{HHb}+\delta C_\mathrm{O_2Hb}\,\varepsilon_\mathrm{O_2Hb}+\delta f_\text{lip}\big(\mu_{a,\text{lip}}-\mu_{a,w}\big)\Big]
$$

dove si è usata la **complementarità** lipidi/acqua $\delta f_w=-\delta f_\text{lip}$: se aumenta il grasso diminuisce l'acqua. Le due emoglobine sono assegnate separatamente; da esse si ricavano emoglobina totale e saturazione dell'inclusione:

$$
\text{tHb}=C_\text{HHb}+C_\mathrm{O_2Hb},\qquad \text{SO}_2=\frac{C_\mathrm{O_2Hb}}{\text{tHb}}.
$$

**Implementazione.** `delta_mua_inclusione` costruisce il dizionario `d_comp` delle variazioni (con `"H2O": -d_lip`) e riusa `mua_background`: è la linearità della formula tradotta in codice. I `raise ValueError` bloccano composizioni non fisiche (frazioni fuori da [0, 1], concentrazioni negative). `ripartisci_tHb` è un aiuto per variazioni a saturazione costante.

In [ ]:
# Variazione dell'inclusione rispetto al background:
#   - lipide/acqua COMPLEMENTARI: dH2O = -d_lip
#   - emoglobina: si assegnano SEPARATAMENTE le variazioni di HHb e di O2Hb
#     [uM]. La variazione di emoglobina totale e' la loro somma,
#         d_tHb = d_HHb + d_O2Hb,
#     e la saturazione dell'inclusione SO2_incl = O2Hb_incl / tHb_incl dice se
#     la variazione e' "ossigenata" (SO2 sale), "deossigenata" (SO2 scende) o
#     a SO2 costante (stessa proporzione del background: vedi ripartisci_tHb).
#     dmu_a = 0.1 [d_HHb eps_HHb + d_O2Hb eps_O2Hb + d_lip (mua_lip - mua_H2O)]
#   Le due emoglobine hanno spettri diversi (HHb ha il picco a 760 nm, O2Hb
#   cresce oltre 800 nm): variare l'una o l'altra cambia la FORMA di dmu_a(lambda).
# !!! LIMITI: |d_lip| <= 0.30;  50 <= |d_tHb| <= 200 uM (oppure 0) !!!
def ripartisci_tHb(d_tHb, comp_bkg):
    """(d_HHb, d_O2Hb) per una variazione d_tHb a SO2 costante (quella del background)."""
    SO2 = comp_bkg["O2Hb"] / (comp_bkg["HHb"] + comp_bkg["O2Hb"])
    return (1 - SO2) * d_tHb, SO2 * d_tHb


def delta_mua_inclusione(spettri, comp_bkg, wl, d_lip=0.0, d_HHb=0.0, d_O2Hb=0.0):
    """dmu_a [mm^-1] (N_wl,) dell'inclusione e sua composizione [dict, con tHb e SO2]."""
    d_tHb = d_HHb + d_O2Hb
    if abs(d_lip) > 0.30:
        print(f"ATTENZIONE: |d_lip| = {abs(d_lip):.2f} > 0.30 (limite: 30%).")
    if d_tHb != 0 and not 50 <= abs(d_tHb) <= 200:
        print(f"ATTENZIONE: |d_tHb| = {abs(d_tHb):.0f} uM fuori dall'intervallo 50-200 uM.")
    d_comp = {"HHb": d_HHb, "O2Hb": d_O2Hb, "Lipid": d_lip, "H2O": -d_lip}
    comp_incl = {k: comp_bkg[k] + d_comp[k] for k in comp_bkg}
    if not (0 <= comp_incl["Lipid"] <= 1 and 0 <= comp_incl["H2O"] <= 1):
        raise ValueError(f"Frazioni non fisiche nell'inclusione: lipide "
                         f"{comp_incl['Lipid']:.2f}, acqua {comp_incl['H2O']:.2f}.")
    if comp_incl["HHb"] < 0 or comp_incl["O2Hb"] < 0:
        raise ValueError(f"Concentrazioni negative nell'inclusione: HHb "
                         f"{comp_incl['HHb']:.1f} uM, O2Hb {comp_incl['O2Hb']:.1f} uM.")
    comp_incl["tHb"] = comp_incl["HHb"] + comp_incl["O2Hb"]
    comp_incl["SO2"] = comp_incl["O2Hb"] / comp_incl["tHb"] if comp_incl["tHb"] > 0 else np.nan
    return mua_background(spettri, d_comp, wl), comp_incl

Calcolo degli spettri di background: due vettori di lunghezza `N_wl`, uno per $\mu_a(\lambda)$ e uno per $\mu_s'(\lambda)$.

In [ ]:
spettri = leggi_spettri(file_spettri, foglio_spettri)
mu_a_bkg = mua_background(spettri, composizione, wl_nm)          # [mm^-1] (N_wl,)
mu_s_bkg = mus_jacques(wl_nm)                                    # [mm^-1] (N_wl,)

<a id="step1"></a>
## 2. Geometria, griglia di voxel e gate temporali (STEP 1)

### Condizione al contorno estrapolata

All'interfaccia tessuto–aria una parte della luce viene riflessa indietro per la differenza di indice di rifrazione. Nell'approssimazione di diffusione questo si traduce nella condizione di Robin (rassegna, eq. 15), che in pratica si impone azzerando la fluenza su un piano **esterno** al mezzo, a distanza

$$
z_b=2AD,\qquad D=\frac{1}{3\mu_s'},\qquad A=\frac{1+R_\text{eff}}{1-R_\text{eff}},
$$

dove $R_\text{eff}(n)$ è la riflettanza effettiva della superficie, calcolata con il fit polinomiale in $n$ di uso comune in letteratura (`A_coefficient`). Con $n=1.4$ si ha $A\approx3.25$.

### Sorgenti e rivelatori

`geometria_base` elenca gli optodi sulla superficie $z=0$. Le *list comprehension* con filtro (`if p["type"] == "source"`) separano sorgenti e rivelatori in due array `(N, 2)`; `np.column_stack` aggiunge la coordinata $z=0$ ai rivelatori.

In [ ]:
# %% =============================================================
# STEP 1 - Geometria sorgenti/rivelatori, griglia, gate
# ==================================================================
n_idx = 1.4           # [-]      indice di rifrazione del mezzo

def A_coefficient(n):
    """Coefficiente A della condizione al contorno estrapolata [-]."""
    Reff = -1.4399/n**2 + 0.7099/n + 0.6681 + 0.0636*n
    return (1 + Reff)/(1 - Reff)

A_coeff = A_coefficient(n_idx)

geometria_base = {
    "unit": "mm",
    "points": [
        {"type": "detector", "x": 0,   "y": 0},
        {"type": "source",   "x": 0,   "y": 12},
        {"type": "source",   "x": -12, "y": 0},
        {"type": "source",   "x": 12,  "y": 0},
        {"type": "source",   "x": 0,   "y": -12},
        {"type": "detector", "x": -12, "y": 12},
        {"type": "detector", "x": 0,   "y": 24},
        {"type": "detector", "x": 12,  "y": 12},
        {"type": "detector", "x": 12,  "y": -12},
        {"type": "detector", "x": -12, "y": -12},
        {"type": "detector", "x": 0,   "y": -24},
        {"type": "detector", "x": 24,  "y": 0},
        {"type": "detector", "x": -24, "y": 0},
        {"type": "source",   "x": -12, "y": 24},
        {"type": "source",   "x": 12,  "y": 24},
        {"type": "source",   "x": 12,  "y": -24},
        {"type": "source",   "x": -12, "y": -24},
        {"type": "detector", "x": -24, "y": 24},
        {"type": "detector", "x": 24,   "y": 24},
        {"type": "detector", "x": 24,  "y": -24},
        {"type": "detector", "x": -24,  "y": -24},
    ],
}

src_xy = np.array([[p["x"], p["y"]] for p in geometria_base["points"]
                   if p["type"] == "source"], dtype=float)
det_xy = np.array([[p["x"], p["y"]] for p in geometria_base["points"]
                   if p["type"] == "detector"], dtype=float)
N_src, N_det = src_xy.shape[0], det_xy.shape[0]
r_d0 = np.column_stack((det_xy, np.full(N_det, 0.0)))           # rivelatori a z = 0

### Coppie sorgente-rivelatore

Non tutte le coppie vengono usate. Per ogni sorgente i rivelatori sono raggruppati in **ordini di vicinanza**: il primo ordine sono quelli alla distanza minima, il secondo quelli alla distanza successiva, e così via. Con `"primi_secondi_vicini"` si ottengono 64 coppie a $\rho=12$ e $26.8$ mm.

**Implementazione** (`ordini_di_vicinanza`):

1. si scorrono le distanze ordinate e si crea un nuovo "livello" solo quando la distanza supera l'ultimo livello di più di `tol`;
2. `np.abs(dist[:, None] - livelli[None, :])` è una matrice `(N_det, N_livelli)` costruita per **broadcasting**: `[:, None]` aggiunge un asse, e `numpy` replica automaticamente i due vettori;
3. `argmin(axis=1) + 1` assegna a ogni rivelatore il livello più vicino, numerato da 1.

In [ ]:
modalita_coppie = "primi_secondi_vicini"
tol_vicini = 0.5
coppie_manuali = [(1, 1), (1, 4), (3, 8)]
ORDINI_PER_MODALITA = {"primi_vicini": 1, "primi_secondi_vicini": 2, "tutte": None}


def ordini_di_vicinanza(dist, tol):
    livelli = []
    for dd in np.sort(dist):
        if not livelli or dd - livelli[-1] > tol:
            livelli.append(dd)
    livelli = np.array(livelli)
    return np.argmin(np.abs(dist[:, None] - livelli[None, :]), axis=1) + 1


def seleziona_coppie(modalita, tol=tol_vicini, manuali=None):
    if modalita == "manuale":
        return [(s - 1, d - 1) for s, d in manuali]
    ordine_max = ORDINI_PER_MODALITA[modalita]
    coppie = []
    for s in range(N_src):
        dist = np.linalg.norm(det_xy - src_xy[s], axis=1)
        ordini = ordini_di_vicinanza(dist, tol)
        scelti = np.arange(N_det) if ordine_max is None else np.where(ordini <= ordine_max)[0]
        coppie += [(s, int(d)) for d in scelti[np.argsort(dist[scelti])]]
    return coppie


pairs = seleziona_coppie(modalita_coppie, manuali=coppie_manuali)
N_pairs = len(pairs)
rho_pairs = np.array([np.linalg.norm(src_xy[s] - det_xy[d]) for s, d in pairs])

### Griglia di voxel

Il dominio $64\times64\times32$ mm è diviso in voxel cubici di lato 4 mm: $16\times16\times8=2048$ voxel. I voxel servono a **discretizzare l'incognita** $\delta\mu_a$ (costante in ogni voxel), non a risolvere l'equazione di diffusione: il modello diretto è analitico.

**Implementazione.**

- `np.arange(-x_ext/2 + step/2, x_ext/2, step)` dà i **centri** dei voxel.
- `np.meshgrid(..., indexing='ij')` crea tre array 3D `X, Y, Z` con indici nell'ordine `[ix, iy, iz]`.
- `X.flatten()` usa l'ordine C (ultimo indice più veloce): il voxel `(ix, iy, iz)` ha indice lineare $j=(i_x N_y+i_y)N_z+i_z$. Per questo, in tutto lo script, `vettore.reshape(Nx, Ny, Nz)` riporta un'immagine in 3D in modo coerente.
- `r_V` ha forma `(2048, 3)`: una riga per voxel.

In [ ]:
step = 4.0
x_ext = 64.0
z_ext = 32.0
x_coords = np.arange(-x_ext/2 + step/2, x_ext/2, step)
y_coords = np.arange(-x_ext/2 + step/2, x_ext/2, step)
z_coords = np.arange(0 + step/2, z_ext, step)
X, Y, Z = np.meshgrid(x_coords, y_coords, z_coords, indexing='ij')
r_V = np.column_stack((X.flatten(), Y.flatten(), Z.flatten()))
N_vox = r_V.shape[0]
V_vox = step**3
Nx, Ny, Nz = x_coords.size, y_coords.size, z_coords.size

### Asse dei tempi e gate

- `t` è l'asse fine (passo 5 ps) su cui si calcolano le funzioni di Green.
- I **gate** sono 20 finestre di 0.2 ns a partire da 0.05 ns. `np.searchsorted(t, g[0])` restituisce l'indice del primo campione $\ge$ inizio del gate: le coppie `(i0, i1)` permettono di integrare con lo slicing `phi[i0:i1]`.
- $N_\text{meas}=N_\text{pairs}\times N_\text{gates}=64\times20=1280$ misure.

**Budget fotonico.** `N_tot_TPSF = 1e8` è il numero di conteggi sull'intera TPSF per coppia. Attenzione alla sintassi: in Python `10^8` è lo XOR bit a bit (vale 2), la potenza si scrive `1e8` o `10**8`.

**Rumore riproducibile.** `seme_rumore` inizializza un unico generatore (`np.random.default_rng`), così lo scan dà sempre gli stessi numeri.

**`modello_sigma`** sceglie da dove viene la deviazione standard usata per pesare le misure (vedi §5): `"baseline"` dai conteggi attesi, una volta per lunghezza d'onda; `"dati"` dai conteggi osservati di ogni misura.

In [ ]:
dt = 0.005
t = np.arange(0.05, 4.1, dt)
t_start = 0.05
gate_width = 0.2
N_gates = 20
gates = [(t_start + gate_width*i, t_start + gate_width*(i + 1)) for i in range(N_gates)]
gate_indices = [(np.searchsorted(t, g[0]), np.searchsorted(t, g[1])) for g in gates]
N_meas = N_pairs * N_gates

# NB: in Python '^' e' lo XOR bit a bit (10^8 = 2): la potenza si scrive 1e8 o 10**8
N_tot_TPSF = 1e8        # conteggi sull'intera TPSF, per coppia s-d
seme_rumore = 42        # rumore riproducibile (un generatore unico per tutto lo scan)

# Sigma usata per pesare le misure (vedi intestazione):
#   "baseline" -> sigma_0 dai conteggi attesi di baseline, una volta per lambda (veloce)
#   "dati"     -> stimata dai conteggi osservati di ogni misura (realistico, piu' lento)
modello_sigma = "baseline"
if modello_sigma not in ("dati", "baseline"):
    raise ValueError(f"modello_sigma = '{modello_sigma}' non valido: usare 'dati' o 'baseline'.")

<a id="step2"></a>
## 3. Inclusioni e variazioni di composizione (STEP 2)

### Teoria

Ogni caso dello scan è un'inclusione cubica di $2\times2\times2$ voxel ($8\times8\times8$ mm) con $\delta\mu_a$ uniforme, in tre posizioni (superficiale, intermedia, profonda), con sei variazioni di composizione:

- V1–V4: ±50 µM di O₂Hb oppure di HHb, una alla volta;
- V5–V6: ±30 punti percentuali di frazione lipidica, con l'acqua complementare.

Il vettore vero è $\mathbf A=\delta\mu_a(\lambda)\,\mathbf 1_\text{incl}$, cioè la maschera dell'inclusione moltiplicata per il $\delta\mu_a$ a quella lunghezza d'onda.

### Implementazione

- La maschera usa disuguaglianze **strette**, `np.abs(X - xp) < L_blk[0]/2`: un voxel appartiene al blocco se il suo centro dista meno di mezzo lato dal centro dell'inclusione. Per un blocco 2×2×2 il centro deve cadere su un vertice fra voxel; il controllo `m.sum() != np.prod(n_blk)` intercetta centri non allineati.
- `posizioni_scan` e `variazioni_scan` (`None` = tutte) filtrano le liste con una *list comprehension*, per scan più brevi.
- `dmu_var[vid]` contiene lo spettro $\delta\mu_a(\lambda)$ di ogni variazione, calcolato una volta sola.

In [ ]:
# %% =============================================================
# STEP 2 - Posizioni e variazioni da scansionare
# ==================================================================
# Inclusione 2x2x2: il centro deve stare su un vertice fra voxel
# (x, y multipli di 4 fra -28 e 28; z in {4, 8, ..., 28}).
n_blk = (2, 2, 2)
posizioni = [   # (id, descrizione, (xp, yp, zp) [mm])
    ("P1", "superficiale", (8.0, 20.0,  8.0)),    # blocco z 4-12 mm
    ("P2", "intermedia",   (8.0,  4.0, 16.0)),    # blocco z 12-20 mm
    ("P3", "profonda",     (0.0,  0.0, 20.0)),    # blocco z 16-24 mm
]

# (id, descrizione, d_lip [-], d_HHb [uM], d_O2Hb [uM])
# Background: HHb 50 uM, O2Hb 150 uM (tHb 200 uM, SO2 75%), lipide 70%, acqua 30%.
# Varia UN cromoforo alla volta, con la stessa ampiezza (+-50 uM): e' l'unica
# ampiezza simmetrica compatibile con i limiti (|d_tHb| >= 50 uM e HHb >= 0;
# HHb -50 porta l'HHb dell'inclusione a 0 uM, il limite fisico).
variazioni = [
    ("V1", "O2Hb +50",  0.0,   0.0,  50.0),   # O2Hb 200 uM, SO2 80%
    ("V2", "O2Hb -50",  0.0,   0.0, -50.0),   # O2Hb 100 uM, SO2 67%
    ("V3", "HHb +50",   0.0,  50.0,   0.0),   # HHb 100 uM,  SO2 60%
    ("V4", "HHb -50",   0.0, -50.0,   0.0),   # HHb 0 uM,    SO2 100%
    # Lipidi +-30 punti percentuali di frazione in volume (il limite |d_lip| <= 0.30),
    # acqua complementare (dH2O = -d_lip), emoglobine invariate:
    ("V5", "lipide +30% (acqua -30%)",  0.30, 0.0, 0.0),   # lipide 100%, acqua 0%
    ("V6", "lipide -30% (acqua +30%)", -0.30, 0.0, 0.0),   # lipide 40%,  acqua 60%
]

# Sottoinsieme da scansionare (None = tutte). Per ridurre i tempi, ad esempio:
# posizioni_scan = ["P2"]; variazioni_scan = ["V3"].
posizioni_scan = None
variazioni_scan = None
if posizioni_scan is not None:
    posizioni = [p for p in posizioni if p[0] in posizioni_scan]
if variazioni_scan is not None:
    variazioni = [v for v in variazioni if v[0] in variazioni_scan]
if not posizioni or not variazioni:
    raise ValueError("Nessuna posizione o variazione selezionata: controllare posizioni_scan "
                     "e variazioni_scan.")

L_blk = np.array(n_blk) * step
maschere = {}
for pid, pdesc, (xp, yp, zp) in posizioni:
    m = ((np.abs(X - xp) < L_blk[0]/2) & (np.abs(Y - yp) < L_blk[1]/2) &
         (np.abs(Z - zp) < L_blk[2]/2)).flatten()
    if m.sum() != np.prod(n_blk):
        raise ValueError(f"{pid}: il blocco contiene {m.sum()} voxel invece di "
                         f"{np.prod(n_blk)} (centro non allineato alla griglia).")
    maschere[pid] = m

dmu_var, comp_var = {}, {}
for vid, vdesc, d_lip, d_HHb, d_O2Hb in variazioni:
    dmu_var[vid], comp_var[vid] = delta_mua_inclusione(spettri, composizione, wl_nm,
                                                       d_lip, d_HHb, d_O2Hb)

N_casi = N_wl * len(posizioni) * len(variazioni)
print("="*78)
print(f"SCAN: {N_wl} lambda ({wl_nm[0]:.0f}-{wl_nm[-1]:.0f} nm) x {len(posizioni)} posizioni x "
      f"{len(variazioni)} variazioni = {N_casi} ricostruzioni")
print(f"Geometria: {N_pairs} coppie s-d ({modalita_coppie}), {N_gates} gate, "
      f"{N_meas} misure, {N_vox} voxel, N_tot = {N_tot_TPSF:.0e} conteggi/TPSF/coppia")
print(f"Pesi delle misure: modello_sigma = '{modello_sigma}' "
      + ("(sigma stimata dai conteggi osservati, SVD per caso)" if modello_sigma == "dati"
         else "(sigma_0 dai conteggi attesi di baseline, SVD per lambda)"))
print("Posizioni: " + "; ".join(f"{p[0]} {p[2]}" for p in posizioni))
print(f"Variazioni (background: HHb {composizione['HHb']:.0f} uM, O2Hb {composizione['O2Hb']:.0f} uM, "
      f"SO2 {100*composizione['O2Hb']/(composizione['HHb']+composizione['O2Hb']):.0f}%, "
      f"lipide {100*composizione['Lipid']:.0f}%):")
for vid, vdesc, d_lip, d_HHb, d_O2Hb in variazioni:
    ci = comp_var[vid]
    print(f"   {vid} {vdesc:38s} dO2Hb {d_O2Hb:+7.1f}  dHHb {d_HHb:+6.1f}  "
          f"dtHb {d_HHb + d_O2Hb:+7.1f} uM  dlip {d_lip:+.2f}  ->  SO2 incl {100*ci['SO2']:.0f}%, "
          f"dmu_a {dmu_var[vid].min():+.1e}..{dmu_var[vid].max():+.1e} mm^-1")
print("="*78 + "\n")

<a id="forward"></a>
## 4. Modello diretto: diffusione, perturbazione di Born, matrice W

### 4.1 Approssimazione di diffusione

La propagazione della luce è descritta esattamente dall'equazione del trasporto radiativo (RTE, rassegna §3.1). Se la luce è già molto diffusa ($\mu_a\ll\mu_s'$, distanze molto maggiori di $1/\mu_s'$, tempi lunghi rispetto al tempo fra due urti) l'RTE si riduce all'**equazione di diffusione** per la fluenza $\Phi(\mathbf r,t)$ (rassegna, eq. 14):

$$
\left(\frac1v\frac{\partial}{\partial t}+\mu_a-\nabla\cdot D\nabla\right)\Phi(\mathbf r,t)=q(\mathbf r,t),\qquad v=\frac{c}{n},\quad D=\frac{1}{3\mu_s'}.
$$

Nel dominio del tempo si usa $D=1/(3\mu_s')$ senza $\mu_a$, scelta comune e lecita perché $\mu_a\ll\mu_s'$.

### 4.2 Funzione di Green nel mezzo infinito

La risposta a un impulso isotropo in un mezzo omogeneo infinito (`fluence_inf`) è

$$
G(r,t)=\frac{v}{(4\pi Dvt)^{3/2}}\exp\!\left(-\frac{r^2}{4Dvt}-\mu_a vt\right).
$$

I due fattori si leggono così: la gaussiana in $r$ descrive la nube di fotoni che si allarga come $\sqrt{Dvt}$ (cammino casuale), e $e^{-\mu_a vt}$ è la legge di Beer–Lambert applicata al cammino percorso $vt$.

### 4.3 Perturbazione di Born

Un voxel $j$ di volume $V$ con assorbimento $\mu_a+\delta\mu_a$ si comporta come una sorgente negativa $-\delta\mu_a\,\Phi_0$. Al primo ordine (Born) la variazione di fluenza al rivelatore è

$$
\delta\Phi(\mathbf r_d,t)=-\,\delta\mu_a V\int_0^t G(r_{jd},t-\tau)\,G(r_{sj},\tau)\,d\tau ,
$$

cioè: la luce va dalla sorgente al voxel in un tempo $\tau$, viene parzialmente assorbita, e il resto va dal voxel al rivelatore nel tempo rimanente. Per due Green di mezzo infinito la convoluzione si calcola analiticamente. I due fattori $e^{-\mu_a v\tau}$ ed $e^{-\mu_a v(t-\tau)}$ danno $e^{-\mu_a vt}$, e l'integrale di Laplace

$$
\int_0^t\frac{e^{-a/(t-\tau)}\,e^{-b/\tau}}{(t-\tau)^{3/2}\tau^{3/2}}\,d\tau=\sqrt\pi\left(\frac1{\sqrt a}+\frac1{\sqrt b}\right)\frac{e^{-(\sqrt a+\sqrt b)^2/t}}{t^{3/2}},\qquad a=\frac{r_{jd}^2}{4Dv},\ b=\frac{r_{sj}^2}{4Dv},
$$

porta alla formula di `deltafluence_inf`:

$$
\delta\Phi=-\frac{v^2\,\delta\mu_a V}{(4\pi Dv)^{5/2}\,t^{3/2}}\left(\frac1{r_{sj}}+\frac1{r_{jd}}\right)\exp\!\left(-\frac{(r_{sj}+r_{jd})^2}{4Dvt}-\mu_a vt\right).
$$

### Implementazione

Le due funzioni sono scritte per lavorare su **array**: se `t` ha forma `(1, N_t)` e le distanze hanno forma `(N_vox, 1)`, il risultato ha forma `(N_vox, N_t)` per broadcasting, cioè la perturbazione di tutti i voxel a tutti i tempi in un'unica chiamata, senza cicli Python.

In [ ]:
# %% =============================================================
# FUNZIONI
# ==================================================================
def deltafluence_inf(mu_a0, mu_s0, t, dmu_a, V, r, r1, n=n_idx):
    D = 1/(3*mu_s0); v = 3e2/n
    return (-(v**2)/((4*pi*D*v)**(5/2)*t**(3/2))*exp(-mu_a0*v*t)
            * dmu_a*V*(1/r + 1/r1)*exp(-(r+r1)**2/(4*D*v*t)))

def fluence_inf(mu_a0, mu_s0, n, t, r):
    D = 1/(3*mu_s0); v = 3e2/n
    return (v/(4*pi*D*v*t)**(3/2))*exp(-(r**2/(4*D*v*t)) - mu_a0*v*t)

### 4.4 Mezzo semi-infinito: sorgenti immagine

Per azzerare la fluenza sul piano estrapolato $z=-z_b$ si usa il **metodo delle immagini**:

- la sorgente reale è posta a profondità $z_0=1/\mu_s'$ (dove il fascio collimato diventa isotropo);
- una sorgente immagine **negativa** è posta simmetricamente rispetto a $z=-z_b$, cioè a $z=-z_0-2z_b$.

La TPSF di riferimento è quindi $\Phi_0(t)=G(r_{s\to d},t)-G(r_{s'\to d},t)$ (`phi0`).

Nella perturbazione servono due tratti, sorgente → voxel e voxel → rivelatore, ciascuno con la sua immagine: si ottengono **quattro termini** con segni alterni,

$$
\delta\Phi_\text{semi}=\delta\Phi(s,d)-\delta\Phi(s,d')-\delta\Phi(s',d)+\delta\Phi(s',d'),
$$

dove $s'$ e $d'$ sono le immagini. Per il secondo tratto si usa l'immagine del *rivelatore* a $z=-2z_b$ al posto di quella del voxel: poiché il rivelatore sta a $z=0$, le due distanze coincidono.

### 4.5 Dalla fluenza alla matrice W

La misura del gate $g=[t_g,t_g+\Delta t]$ è la variazione relativa dei conteggi integrati nel gate. Calcolando la perturbazione per $\delta\mu_a=1$ mm⁻¹:

$$
W_{(p,g),\,j}=\frac{\displaystyle\int_g\delta\Phi_j(t)\,dt}{\displaystyle\int_g\Phi_0(t)\,dt}
\qquad\Longrightarrow\qquad
\mathbf M\approx W\,\mathbf A .
$$

Intuitivamente $-W_{(p,g),j}$ è il **cammino medio** che i fotoni rivelati dalla coppia $p$ nel gate $g$ percorrono dentro il voxel $j$: più tempo i fotoni passano nel voxel, più ne risentono dell'assorbimento. La riga $(p,g)$, riportata in 3D, è la "banana" di sensibilità di quella misura; i gate tardivi pesano voxel più profondi.

La funzione restituisce anche `frac[p, g]`, la frazione della TPSF che cade in ogni gate, che serve a calcolare i conteggi attesi.

### Implementazione (`matrice_W`)

- Le distanze sono precalcolate per tutti i voxel e tutte le sorgenti in un colpo: `r_V[None] - r_s0[:, None]` ha forma `(N_src, N_vox, 3)` e `np.linalg.norm(..., axis=2)` dà la matrice delle distanze `(N_src, N_vox)`.
- Nel ciclo sulle coppie, `[:, None]` trasforma le distanze in colonne `(N_vox, 1)` e `tt = t[None, :]` è una riga: `dphi` ha forma `(N_vox, N_t)`.
- L'integrale nel gate è una somma di Riemann: `np.sum(dphi[:, i0:i1], axis=1) * dt`.
- La riga di `W` ha indice `p*N_gates + g`: è lo stesso ordine che si ottiene con `M.reshape(N_pairs, N_gates).flatten()`.

In [ ]:
def matrice_W(mu_a0, mu_s0):
    """Matrice di sensitivita' W e frazione della TPSF in ogni gate."""
    D = 1/(3*mu_s0); z0 = 1/mu_s0; zb = 2*A_coeff*D
    r_s0 = np.column_stack((src_xy, np.full(N_src, z0)))
    r_si = np.column_stack((src_xy, np.full(N_src, -z0 - 2*zb)))
    r_di = np.column_stack((det_xy, np.full(N_det, -2*zb)))
    r_sv_all  = np.linalg.norm(r_V[None] - r_s0[:, None], axis=2)
    r_siv_all = np.linalg.norm(r_V[None] - r_si[:, None], axis=2)
    r_vd_all  = np.linalg.norm(r_V[None] - r_d0[:, None], axis=2)
    r_vdi_all = np.linalg.norm(r_V[None] - r_di[:, None], axis=2)
    W = np.zeros((N_meas, N_vox))
    frac = np.zeros((N_pairs, N_gates))
    tt = t[None, :]
    for p, (s, d) in enumerate(pairs):
        a, b = r_sv_all[s][:, None], r_siv_all[s][:, None]
        c, e = r_vd_all[d][:, None], r_vdi_all[d][:, None]
        dphi = (deltafluence_inf(mu_a0, mu_s0, tt, 1.0, V_vox, a, c)
                - deltafluence_inf(mu_a0, mu_s0, tt, 1.0, V_vox, a, e)
                - deltafluence_inf(mu_a0, mu_s0, tt, 1.0, V_vox, b, c)
                + deltafluence_inf(mu_a0, mu_s0, tt, 1.0, V_vox, b, e))
        phi0 = (fluence_inf(mu_a0, mu_s0, n_idx, t, np.linalg.norm(r_d0[d] - r_s0[s]))
                - fluence_inf(mu_a0, mu_s0, n_idx, t, np.linalg.norm(r_d0[d] - r_si[s])))
        tot = np.sum(phi0) * dt
        for g, (i0, i1) in enumerate(gate_indices):
            int_phi0 = np.sum(phi0[i0:i1]) * dt
            W[p*N_gates + g, :] = np.sum(dphi[:, i0:i1], axis=1) * dt / int_phi0
            frac[p, g] = int_phi0 / tot
    return W, frac

<a id="rumore"></a>
## 5. Modello del rumore e sbiancamento delle misure

### 5.1 Conteggi e misura relativa

Con TCSPC il dato è un conteggio di fotoni, distribuito secondo Poisson. Per la coppia $p$ e il gate $g$:

$$
N_0\sim\text{Poisson}(\lambda_0),\quad N\sim\text{Poisson}\big(\lambda_0(1+M)\big),\qquad \lambda_0=N_\text{tot}\,\frac{\int_g\Phi_0\,dt}{\int\Phi_0\,dt},
$$

con baseline $N_0$ e misura perturbata $N$ acquisite indipendentemente. La misura stimata è

$$
\hat M=\frac{N}{N_0+1}-1,
$$

dove il $+1$ evita la divisione per zero quando un gate non riceve fotoni.

### 5.2 Varianza con il metodo delta

Per il rapporto di due variabili indipendenti, al primo ordine,

$$
\operatorname{Var}\!\left(\frac{N}{N_0}\right)\approx\left(\frac{\partial}{\partial N}\frac{N}{N_0}\right)^2\operatorname{Var}N+\left(\frac{\partial}{\partial N_0}\frac{N}{N_0}\right)^2\operatorname{Var}N_0=\frac{N}{N_0^2}+\frac{N^2}{N_0^3},
$$

usando $\operatorname{Var}N=N$ (Poisson). Lo script calcola due versioni:

- **dai conteggi osservati** (`sig_hat`, modalità `"dati"`): la formula con $n_\text{pert}+1$ e $n_0+1$, che è ciò che si può fare con dati reali;
- **dai conteggi attesi di baseline** (`sig0`, modalità `"baseline"`): con $N\approx N_0\approx\lambda_0$ la formula si riduce a $\sigma_0=\sqrt{2/(\lambda_0+1)}$. Non dipende né dall'inclusione né dal rumore estratto, quindi è la stessa per tutti i casi di una lunghezza d'onda.

### 5.3 Sbiancamento

I gate vicini al picco della TPSF hanno milioni di conteggi, quelli tardivi poche decine: le varianze differiscono di ordini di grandezza. La stima a massima verosimiglianza con rumore gaussiano indipendente minimizza il $\chi^2$, cioè ogni residuo diviso per la propria deviazione standard:

$$
\chi^2=\sum_m\frac{\big(M_m-(W\mathbf A)_m\big)^2}{\sigma_m^2}=\big\|W_w\mathbf A-\mathbf m_w\big\|^2,\qquad
W_w=\operatorname{diag}(1/\sigma_m)\,W,\quad \mathbf m_w=\mathbf M/\boldsymbol\sigma .
$$

Dopo la divisione ogni componente di $\mathbf m_w$ ha rumore a varianza 1 (i dati sono "sbiancati"). Tutte le grandezze successive (soglie, $\lambda$, segnali) sono quindi espresse in **unità di deviazione standard del rumore**.

### Implementazione (`simula_misura`)

- `rng.poisson(lam0)` estrae tutti i conteggi `(N_pairs, N_gates)` insieme.
- `np.clip(..., 0, None)` evita medie negative se $M<-1$ (non succede con perturbazioni fisiche, ma protegge la funzione).
- `.astype(float)` evita overflow negli interi quando si calcolano potenze dei conteggi.

In [ ]:
def simula_misura(lam0, M_ideal, rng):
    """Baseline + misura perturbata (Poisson). Restituisce M stimato e la sua
    deviazione standard stimata dai conteggi osservati (metodo delta):
        Var(N/N0) ~ N/N0^2 + N^2/N0^3   (+1 per evitare divisioni per zero)."""
    n0 = rng.poisson(lam0).astype(float)
    npert = rng.poisson(np.clip(lam0 * (1 + M_ideal), 0, None)).astype(float)
    M_est = npert / (n0 + 1) - 1
    sig_hat = np.sqrt((npert + 1) / (n0 + 1)**2 + (npert + 1)**2 / (n0 + 1)**3)
    return M_est, sig_hat

<a id="tikhonov"></a>
## 6. TSVD e Tikhonov

### 6.1 La decomposizione ai valori singolari

$$
W_w=U\,\operatorname{diag}(s_k)\,V^T,\qquad s_1\ge s_2\ge\dots\ge0 ,
$$

con colonne $\mathbf u_k$ (modi nello spazio delle misure) e $\mathbf v_k$ (modi nello spazio dei voxel) ortonormali. Proiettando le misure sui modi, $c_k=\mathbf u_k^T\mathbf m_w$, la soluzione ai minimi quadrati sarebbe

$$
\hat{\mathbf A}=\sum_k\frac{c_k}{s_k}\,\mathbf v_k .
$$

**Perché il problema è mal posto.** I valori singolari decadono di molti ordini di grandezza. Ogni $c_k$ contiene rumore di varianza 1 (le misure sono sbiancate e $U$ è ortonormale), quindi per i modi con $s_k$ piccolo il termine $c_k/s_k$ amplifica il rumore in modo esplosivo. Le regolarizzazioni lineari moltiplicano ogni modo per un **fattore di filtro** $f_k\in[0,1]$:

$$
\hat{\mathbf A}=\sum_k f_k\,\frac{c_k}{s_k}\,\mathbf v_k .
$$

### 6.2 TSVD

Filtro a gradino: $f_k=1$ per $k\le K$, $0$ altrimenti. Poiché $U$ è ortonormale, il residuo dopo $K$ modi è la parte dei dati non spiegata da quei modi:

$$
\chi^2(K)=\big\|\mathbf m_w-U_KU_K^T\mathbf m_w\big\|^2=\|\mathbf m_w\|^2-\sum_{k\le K}c_k^2 .
$$

### 6.3 Tikhonov

Si aggiunge una penalità quadratica:

$$
\hat{\mathbf A}=\arg\min_{\mathbf A}\|W_w\mathbf A-\mathbf m_w\|^2+\lambda^2\|\mathbf A\|^2 .
$$

Ponendo a zero il gradiente, $2W_w^T(W_w\mathbf A-\mathbf m_w)+2\lambda^2\mathbf A=0$, si ottengono le equazioni normali

$$
\big(W_w^TW_w+\lambda^2I\big)\hat{\mathbf A}=W_w^T\mathbf m_w .
$$

Sostituendo la SVD, $W_w^TW_w=V\operatorname{diag}(s_k^2)V^T$ e $W_w^T\mathbf m_w=V\operatorname{diag}(s_k)\,\mathbf c$, il sistema diventa diagonale nella base $\mathbf v_k$:

$$
\hat{\mathbf A}=\sum_k\frac{s_k\,c_k}{s_k^2+\lambda^2}\,\mathbf v_k=\sum_k\underbrace{\frac{s_k^2}{s_k^2+\lambda^2}}_{f_k}\frac{c_k}{s_k}\,\mathbf v_k .
$$

È un troncamento "morbido": i modi con $s_k\gg\lambda$ passano, quelli con $s_k\ll\lambda$ vengono attenuati di circa $s_k^2/\lambda^2$. Il residuo ha due parti: quella dei dati fuori dall'immagine di $W_w$ (`res_fuori`, che nessun $\mathbf A$ può spiegare) e quella dei modi attenuati,

$$
\chi^2(\lambda)=\underbrace{\Big(\|\mathbf m_w\|^2-\sum_kc_k^2\Big)}_{\texttt{res\_fuori}}+\sum_k\big[(1-f_k)\,c_k\big]^2 .
$$

### 6.4 Principio di discrepanza

Se il modello è giusto e il rumore sbiancato ha varianza 1, il $\chi^2$ della soluzione corretta vale in media $N_\text{meas}$. Scendere sotto vuol dire fittare il rumore, restare sopra vuol dire non spiegare i dati. Si sceglie quindi:

- TSVD: il più piccolo $K$ con $\chi^2(K)\le N_\text{meas}$;
- Tikhonov: il $\lambda$ con $\chi^2(\lambda)=N_\text{meas}$. Poiché $\chi^2$ cresce con $\lambda$, lo si trova per bisezione su $\ln\lambda$.

Nel linguaggio bayesiano, Tikhonov è la stima MAP con prior gaussiano $\mathbf A\sim\mathcal N(0,\lambda^{-2}I)$: la discrepanza sceglie implicitamente l'ampiezza tipica delle variazioni, $1/\lambda$.

### Implementazione

- La SVD (`np.linalg.svd(W_w, full_matrices=False)`, forma "economica") è calcolata una volta per lunghezza d'onda in `prepara_pesi`; per ogni caso basta il prodotto `c = Uw.T @ m_w`.
- TSVD: `np.cumsum(c**2)` dà in un colpo i $\chi^2$ per tutti i $K$; `np.where(chi2 <= N_meas)[0]` trova i $K$ ammessi e si prende il primo.
- Tikhonov: `chi2` è una funzione anonima (`lambda lam: ...`) valutata nella bisezione. La ricerca è su $\ln\lambda\in[\ln(10^{-10}s_1),\ln(10\,s_1)]$, con 60 dimezzamenti. Se il $\chi^2$ resta sopra $N_\text{meas}$ anche con $\lambda$ minimo, o sotto con $\lambda$ massimo, la funzione restituisce l'estremo: con dati reali questo segnala un errore di modello o una $\sigma$ sbagliata (si controlla con la colonna `chi2_finale`).
- La soluzione è `Vtw.T @ (f * c / sw)`: i filtri moltiplicano elemento per elemento, `Vtw.T` ricombina i modi.

In [ ]:
# --- TSVD / Tikhonov (SVD di W_w: per lambda o per caso, vedi modello_sigma) --
def tsvd_disc(c, m_w):
    """TSVD con k per discrepanza: il piu' piccolo k con chi^2 <= N_meas."""
    chi2 = np.sum(m_w**2) - np.cumsum(c**2)
    ok = np.where(chi2 <= N_meas)[0]
    k = int(ok[0]) + 1 if ok.size else sw.size
    return Vtw[:k].T @ (c[:k] / sw[:k]), k, float(chi2[k - 1])

def tikhonov_disc(c, m_w):
    """Tikhonov con lam per discrepanza (bisezione in log(lam)): chi^2 = N_meas."""
    res_fuori = max(np.sum(m_w**2) - np.sum(c**2), 0.0)
    chi2 = lambda lam: res_fuori + np.sum(((1 - sw**2/(sw**2 + lam**2)) * c)**2)
    lo, hi = np.log(1e-10*sw[0]), np.log(10*sw[0])
    if chi2(np.exp(lo)) > N_meas:
        lam = np.exp(lo)
    elif chi2(np.exp(hi)) < N_meas:
        lam = np.exp(hi)
    else:
        for _ in range(60):
            mid = 0.5*(lo + hi)
            if chi2(np.exp(mid)) > N_meas:
                hi = mid
            else:
                lo = mid
        lam = np.exp(0.5*(lo + hi))
    f = sw**2/(sw**2 + lam**2)
    return Vtw.T @ (f * c / sw), lam, float(chi2(lam))

<a id="l1"></a>
## 7. Regolarizzazione L1 con FISTA

### 7.1 Perché la norma 1

Tikhonov penalizza $\sum_jA_j^2$. A parità di segnale sulle misure conviene **spalmare** il valore su molti voxel: mettere 1 in un voxel costa $1^2=1$, mettere 0.5 in quattro voxel equivalenti costa $4\times0.25=1$, e se i voxel vicini sono quasi equivalenti per le misure (sempre vero in DOT) la soluzione spalmata vince. Il risultato è un'immagine sfocata e **sottostimata** (rassegna §4.1).

La norma 1 penalizza invece $\sum_j|A_j|$. Con due incognite e una sola misura (rassegna, Fig. 2), l'insieme delle soluzioni compatibili è una retta. Gonfiando la "palla" della norma finché tocca la retta:

- per $p=2$ la palla è un cerchio e tocca la retta in un punto generico, con entrambe le componenti diverse da zero;
- per $p=1$ la palla è un rombo con i vertici sugli assi, e quasi sempre tocca la retta su un vertice: una componente è **esattamente zero**.

È l'origine della **sparsità**, adatta a un'inclusione piccola in un fondo omogeneo (8 voxel su 2048). In termini bayesiani, la norma 1 corrisponde a un prior di Laplace $p(\mathbf A)\propto e^{-\lambda\sum_j n_j|A_j|}$: quasi tutti i voxel a zero, pochi grandi, di segno qualsiasi.

### 7.2 Il problema pesato e la compensazione di profondità

Lo script risolve

$$
\hat{\mathbf A}=\arg\min_{\mathbf A}\;\tfrac12\|W_w\mathbf A-\mathbf m_w\|^2+\lambda\sum_jn_j|A_j|,\qquad n_j=\|\mathbf w_j\|_2 ,
$$

dove $\mathbf w_j$ è la colonna $j$ di $W_w$, cioè l'**impronta** del voxel $j$ sulle misure. Le quantità in gioco:

- $n_j$ è la **visibilità** del voxel: il segnale, in deviazioni standard del rumore, che produrrebbe una variazione $\delta\mu_a=1$ mm⁻¹ nel solo voxel $j$. Un voxel superficiale vicino agli optodi ha $n_j$ grande; uno profondo o lontano, piccolo.
- Il peso $n_j$ realizza la **compensazione di profondità**. Senza di esso la penalità costerebbe lo stesso per ogni mm⁻¹ di $\delta\mu_a$: per spiegare un dato segnale un voxel superficiale richiede un $A_j$ piccolo e uno profondo un $A_j$ grande, quindi l'ottimizzatore sceglierebbe sempre il superficiale e le inclusioni profonde verrebbero ricostruite vicino alla superficie. Con il peso si penalizza il **segnale prodotto** $n_j|A_j|$, e superficie e profondità competono alla pari.
- Si escludono dalle incognite i voxel **non osservabili**, quelli che anche con la massima variazione plausibile $\Delta\mu_\text{max}=0.05$ mm⁻¹ produrrebbero meno di una deviazione standard:
$$
\mathcal O=\{\,j:\ n_j\,\Delta\mu_\text{max}\ge1\,\}\qquad(\texttt{oss\_L1}).
$$
Per loro $1/n_j$ è enorme, e un valore minuscolo della variabile normalizzata diventerebbe un artefatto enorme in $\delta\mu_a$.

### 7.3 Cambio di variabile

Si passa alla variabile "segnale in sigma":

$$
x_j=n_jA_j,\qquad B=W_w[:,\mathcal O]\,\operatorname{diag}(1/n_j),\qquad \mathbf m=\mathbf m_w .
$$

Le colonne di $B$ hanno norma 1 (`B_L1`), e il problema diventa un LASSO standard:

$$
J(\mathbf x)=\underbrace{\tfrac12\|B\mathbf x-\mathbf m\|^2}_{g(\mathbf x):\ \text{liscia}}+\underbrace{\lambda\|\mathbf x\|_1}_{h(\mathbf x):\ \text{non liscia}} .
$$

La parte $g$ è quadratica, con gradiente ben definito. La parte $h$ ha uno **spigolo** in ogni $x_j=0$: lì la derivata vale $-1$ da sinistra e $+1$ da destra. Per questo la condizione "gradiente = 0" non si può usare e non esiste una formula chiusa come per Tikhonov.

### 7.4 Condizioni di ottimalità: subdifferenziale e regola di Fermat

**Subgradiente.** Per una funzione convessa $f$, un numero $\gamma$ è un subgradiente in $a_0$ se la retta di pendenza $\gamma$ per $(a_0,f(a_0))$ sta tutta sotto il grafico:

$$
f(a)\ge f(a_0)+\gamma\,(a-a_0)\qquad\forall a .
$$

Il **subdifferenziale** $\partial f(a_0)$ è l'insieme di tutti i subgradienti. Dove $f$ è derivabile contiene solo la derivata; in uno spigolo contiene tutte le pendenze fra la derivata sinistra e quella destra. Per il valore assoluto:

$$
\partial|a|=\begin{cases}\{\operatorname{sign}(a)\}&a\ne0\\[2pt][-1,\,1]&a=0 .\end{cases}
$$

**Regola di Fermat.** Per una funzione convessa, $a^\star$ è minimo globale **se e solo se** $0\in\partial f(a^\star)$: la retta orizzontale tocca il grafico da sotto. Dove $f$ è derivabile è la solita condizione $f'(a^\star)=0$. Per funzioni convesse non esistono minimi locali che non siano globali.

**Applicazione.** Il gradiente della parte liscia è

$$
\nabla g(\mathbf x)=B^T(B\mathbf x-\mathbf m)=-B^T\mathbf r,\qquad \mathbf r=\mathbf m-B\mathbf x ,
$$

e per una somma "liscia + non liscia" vale $\partial J=\nabla g+\lambda\,\partial\|\cdot\|_1$. Imponendo $0\in\partial J(\hat{\mathbf x})$ componente per componente, con $\mathbf b_j$ la colonna $j$ di $B$:

$$
\hat x_j\ne0:\quad \mathbf b_j^T\mathbf r=\lambda\,\operatorname{sign}(\hat x_j)\qquad\qquad
\hat x_j=0:\quad \big|\mathbf b_j^T\mathbf r\big|\le\lambda .
$$

$\mathbf b_j^T\mathbf r$ è la correlazione fra il residuo e l'impronta normalizzata del voxel: quanto del segnale **non ancora spiegato** si può attribuire al voxel $j$, in deviazioni standard. Le condizioni dicono che un voxel è spento finché questa evidenza non supera $\lambda$ in modulo, e che un voxel acceso "paga" esattamente $\lambda$. Quali voxel siano accesi, e con quale segno, non si sa in anticipo: le combinazioni possibili sono $3^{N}$ (spento, positivo, negativo), da cui la necessità di un algoritmo iterativo.

### 7.5 L'esempio scalare: la soglia morbida

Si consideri un solo voxel:

$$
F(a)=\tfrac12(a-v)^2+\tau|a|,\qquad \partial F(a)=a-v+\tau\,\partial|a| .
$$

- Se $a>0$: $F'(a)=a-v+\tau=0\Rightarrow a=v-\tau$, accettabile solo se $v>\tau$.
- Se $a<0$: $F'(a)=a-v-\tau=0\Rightarrow a=v+\tau$, accettabile solo se $v<-\tau$.
- Se $a=0$: $\partial F(0)=[-v-\tau,\,-v+\tau]$ contiene lo zero se e solo se $|v|\le\tau$.

I tre casi insieme definiscono l'operatore di **soglia morbida** (*soft thresholding*):

$$
\mathcal S_\tau(v)=\operatorname{sign}(v)\max\big(|v|-\tau,\,0\big)=\begin{cases}v-\tau&v>\tau\\0&|v|\le\tau\\v+\tau&v<-\tau .\end{cases}
$$

Due effetti: dentro la "zona morta" $|v|\le\tau$ il voxel viene **azzerato** (sparsità); fuori, viene **avvicinato a zero** di $\tau$ mantenendo il segno (*shrinkage*). Lo shrinkage fa sì che la L1 sottostimi le ampiezze: per questo lo script, dopo aver trovato il supporto, rifà un fit senza penalità (§9).

### 7.6 Gradiente prossimale (ISTA)

**Costante di Lipschitz.** La curvatura massima di $g$ è il massimo autovalore di $B^TB$:

$$
L=\lambda_\text{max}(B^TB)=\|B\|_2^2\qquad(\texttt{Lip\_L1}).
$$

**Maggiorante quadratico.** Poiché nessuna direzione ha curvatura maggiore di $L$, attorno al punto corrente $\mathbf x_0$ la parte liscia sta sotto una parabola isotropa:

$$
g(\mathbf x)\le g(\mathbf x_0)+\nabla g(\mathbf x_0)^T(\mathbf x-\mathbf x_0)+\frac L2\|\mathbf x-\mathbf x_0\|^2 .
$$

**Completamento del quadrato.** Il lato destro si riscrive raccogliendo i termini in $\mathbf x$:

$$
\frac L2\|\mathbf x-\mathbf x_0\|^2+\nabla g^T(\mathbf x-\mathbf x_0)
=\frac L2\Big\|\mathbf x-\underbrace{\Big(\mathbf x_0-\tfrac1L\nabla g(\mathbf x_0)\Big)}_{\mathbf z}\Big\|^2-\frac1{2L}\|\nabla g(\mathbf x_0)\|^2 .
$$

Il secondo termine non dipende da $\mathbf x$. Minimizzare il modello locale "parabola + penalità esatta" equivale quindi a

$$
\mathbf x^\text{new}=\arg\min_{\mathbf x}\;\frac L2\|\mathbf x-\mathbf z\|^2+\lambda\|\mathbf x\|_1=\operatorname{prox}_{h/L}(\mathbf z),\qquad \mathbf z=\mathbf x_0+\frac1LB^T\mathbf r .
$$

**Separabilità.** Dividendo per $L$, il problema è $\sum_j\big[\frac12(x_j-z_j)^2+\frac\lambda L|x_j|\big]$: una somma di problemi scalari indipendenti, ciascuno uguale all'esempio del §7.5 con $v=z_j$ e $\tau=\lambda/L$. Quindi

$$
\boxed{\;\mathbf x^\text{new}=\mathcal S_{\lambda/L}\Big(\mathbf x_0+\tfrac1LB^T(\mathbf m-B\mathbf x_0)\Big)\;}
$$

Ogni iterazione fa tre cose:

1. **confronto con le misure**: $\mathbf r=\mathbf m-B\mathbf x_0$, il residuo in unità di rumore;
2. **retroproiezione**: $\mathbf z=\mathbf x_0+B^T\mathbf r/L$, cioè si aggiunge a ogni voxel la parte di residuo che gli somiglia;
3. **soglia morbida**: si spengono i voxel con $|z_j|\le\lambda/L$ e si riducono gli altri.

Il modello locale coincide con $J$ in $\mathbf x_0$ e sta sopra $J$ ovunque. Ne segue $J(\mathbf x^\text{new})\le\text{modello}(\mathbf x^\text{new})\le\text{modello}(\mathbf x_0)=J(\mathbf x_0)$: il costo non può salire.

In termini della variabile fisica, con $z_j$ riportato in $\delta\mu_a$:

$$
\hat A_j^{(k+1)}=\operatorname{sign}(\tilde z_j)\max\!\Big(|\tilde z_j|-\frac{\lambda}{L\,n_j},0\Big),\qquad \tilde z_j=Y_j+\frac{\mathbf w_j^T(\mathbf m_w-W_w\mathbf Y)}{L\,n_j^2} .
$$

La soglia in $\delta\mu_a$ è $\lambda/(Ln_j)$: più alta per i voxel poco visibili, ma uguale per tutti in termini di segnale prodotto.

### 7.7 Accelerazione FISTA

ISTA converge lentamente, come $J(\mathbf x_k)-J^\star=\mathcal O(1/k)$. FISTA (Beck e Teboulle, 2009) costruisce il modello non nell'ultima stima ma in un punto **spinto in avanti** lungo la direzione di movimento (momento di Nesterov):

$$
\begin{aligned}
\mathbf x^{(k+1)}&=\mathcal S_{\lambda/L}\Big(\mathbf y^{(k)}-\tfrac1LB^T\big(B\mathbf y^{(k)}-\mathbf m\big)\Big)\\
t_{k+1}&=\frac{1+\sqrt{1+4t_k^2}}{2},\qquad t_1=1\\
\mathbf y^{(k+1)}&=\mathbf x^{(k+1)}+\underbrace{\frac{t_k-1}{t_{k+1}}}_{\beta_k}\big(\mathbf x^{(k+1)}-\mathbf x^{(k)}\big)
\end{aligned}
$$

Il coefficiente $\beta_k$ parte da 0 e tende a 1: l'algoritmo "ricorda" la direzione in cui si stava muovendo. Il costo per iterazione è identico a ISTA (un prodotto per $B$ e uno per $B^T$), ma la convergenza diventa $\mathcal O(1/k^2)$. Il costo non è più garantito decrescente a ogni passo (il momento può superare leggermente il minimo), ma lo è la velocità complessiva.

### 7.8 $\lambda_\text{max}$: quando l'immagine è vuota

In $\mathbf x=0$ il residuo coincide con i dati, $\mathbf r=\mathbf m$. Dalle condizioni del §7.4, $\mathbf x=0$ è ottimo se e solo se $|\mathbf b_j^T\mathbf m|\le\lambda$ per ogni $j$:

$$
\lambda_\text{max}=\|B^T\mathbf m\|_\infty=\max_j\frac{|\mathbf w_j^T\mathbf m_w|}{n_j} .
$$

La quantità $\zeta_j=\mathbf w_j^T\mathbf m_w/n_j$ ha un significato preciso: è la **significatività** del fit che spiega tutte le misure con il solo voxel $j$. Infatti il fit a un voxel è $\tilde A_j=\mathbf w_j^T\mathbf m_w/n_j^2$ con deviazione standard $1/n_j$, e $\zeta_j=\tilde A_j/\sigma(\tilde A_j)$. È anche il rapporto segnale/rumore di un filtro adattato all'impronta del voxel. Quindi:

- per $\lambda\ge\lambda_\text{max}$ nessun voxel ha evidenza sufficiente e l'immagine è vuota;
- scendendo sotto $\lambda_\text{max}$ si accende per primo il voxel più significativo;
- scrivere $\lambda=f\cdot\lambda_\text{max}$ rende la scelta indipendente dall'ampiezza dei dati.

Con solo rumore gli $\zeta_j$ sono gaussiani standard e il loro massimo su $N$ voxel vale circa $\sqrt{2\ln N}$ (soglia universale di Donoho e Johnstone): se la discrepanza porta $\lambda$ molto sotto questo valore, è facile accendere voxel di solo rumore.

### 7.9 Scelta di $\lambda$: discrepanza e warm start

1. Si parte da $\lambda=0.3\,\lambda_\text{max}$ e si risolve con FISTA.
2. Se $\chi^2=\|B\hat{\mathbf x}-\mathbf m\|^2\le N_\text{meas}+2\sqrt{2N_\text{meas}}$ ci si ferma. $N_\text{meas}$ è la media del $\chi^2$ del solo rumore e $\sqrt{2N_\text{meas}}$ la sua deviazione standard: si accetta il primo $\lambda$ il cui residuo è compatibile con il rumore entro 2σ.
3. Altrimenti si passa al $\lambda$ successivo della griglia `fattori_L1` (passi di circa un fattore 3) ripartendo dalla soluzione appena trovata (**warm start**, o *continuazione*). Per $\lambda$ grande la soluzione è molto sparsa e si trova in fretta; da lì ci si sposta gradualmente.

Alla fine si torna alla variabile fisica, $\hat A_j=\hat x_j/n_j$ per $j\in\mathcal O$ e $0$ altrove.

### Implementazione: parametri

- `dmu_a_max` definisce i voxel osservabili ($n_j\,\Delta\mu_\text{max}\ge1$).
- `fattori_L1` è la griglia di $f=\lambda/\lambda_\text{max}$ percorsa dalla discrepanza.
- `tol_fista` e `n_iter_max` controllano l'arresto di FISTA: ci si ferma quando $\|\mathbf x^{(k+1)}-\mathbf x^{(k)}\|\le$ `tol_fista` $\cdot\|\mathbf x^{(k+1)}\|$. Portarla a `1e-4` accelera lo scan senza effetti apprezzabili, perché le ampiezze vengono poi rifittate.

### `norma2_quadra`: la costante di Lipschitz con il metodo delle potenze

Calcolare $\|B\|_2^2$ con una SVD costerebbe quanto la SVD di $W_w$. Il **metodo delle potenze** è molto più economico: ripetendo $\mathbf x\leftarrow B^TB\mathbf x/\|B^TB\mathbf x\|$, la componente lungo l'autovettore dominante cresce più delle altre di un fattore $(\lambda_1/\lambda_2)$ a ogni passo, e $\|B^TB\mathbf x\|$ converge all'autovalore massimo. Ogni passo costa due prodotti matrice-vettore.

- `B.T @ (B @ x)` evita di formare la matrice $B^TB$.
- Il ciclo si ferma quando la stima cambia meno di `tol` in relativo.
- Il `margine` dell'1% protegge da una stima leggermente per difetto: con un passo $1/L$ troppo lungo FISTA può divergere, con uno un po' più corto rallenta appena.

In [ ]:
# --- L1 senza vincolo di segno (FISTA) ---------------------------------------
dmu_a_max = 0.05      # [mm^-1] voxel osservabili: ||w_j|| * dmu_a_max >= 1
fattori_L1 = (0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 3e-4, 1e-4)
tol_fista = 1e-5      # arresto: variazione relativa di x < tol_fista
n_iter_max = 2000

def norma2_quadra(B, tol=1e-6, n_iter=300, margine=1.01):
    """||B||_2^2 (autovalore massimo di B^T B) col metodo delle potenze.
    Il margine dell'1% garantisce un passo FISTA stabile anche se la stima
    e' leggermente per difetto."""
    x = np.random.default_rng(0).standard_normal(B.shape[1])
    x /= np.linalg.norm(x)
    val = 0.0
    for _ in range(n_iter):
        y = B.T @ (B @ x)
        val_new = np.linalg.norm(y)
        x = y / val_new
        if abs(val_new - val) <= tol * val_new:
            break
        val = val_new
    return margine * val_new

### `fista_l1`: l'iterazione, riga per riga

| Riga di codice | Formula |
|---|---|
| `x = np.zeros(...) if x0 is None else x0.copy()` | punto di partenza: zero, oppure la soluzione del $\lambda$ precedente (warm start). `.copy()` evita di modificare l'array del chiamante |
| `y, tk = x.copy(), 1.0` | $\mathbf y^{(1)}=\mathbf x^{(1)}$, $t_1=1$ |
| `z = y - B.T @ (B @ y - m) / Lip` | passo di gradiente: $\mathbf z=\mathbf y-\frac1LB^T(B\mathbf y-\mathbf m)$ |
| `x_new = np.sign(z) * np.maximum(np.abs(z) - lam / Lip, 0)` | soglia morbida $\mathcal S_{\lambda/L}(\mathbf z)$, vettorializzata su tutti i voxel |
| `t_new = (1 + np.sqrt(1 + 4*tk*tk)) / 2` | $t_{k+1}$ |
| `y = x_new + (tk - 1) / t_new * (x_new - x)` | punto spinto in avanti con $\beta_k=(t_k-1)/t_{k+1}$ |
| `if np.linalg.norm(x_new - x) <= tol_fista * max(...)` | criterio d'arresto relativo; `max(..., 1e-30)` evita la divisione per zero se la soluzione è nulla |

### `ricostruzione_L1`: discrepanza e ritorno all'immagine

- `lam_max = np.abs(B_L1.T @ m_w).max()` è $\|B^T\mathbf m\|_\infty$.
- `target = N_meas + 2*np.sqrt(2*N_meas)` è la soglia di accettazione del $\chi^2$.
- Il ciclo `for f in fattori_L1` passa `x0=x`: è il warm start.
- `A_out[oss_L1] = x / col_L1[oss_L1]` riporta la soluzione in $\delta\mu_a$ (mm⁻¹) usando l'**indicizzazione booleana**: solo i voxel osservabili ricevono un valore, gli altri restano a zero.
- La funzione restituisce anche il fattore $f$ finale e il $\chi^2$, salvati nell'Excel come `valore_reg` e `chi2_finale`.

In [ ]:
def fista_l1(B, m, lam, Lip, x0=None):
    x = np.zeros(B.shape[1]) if x0 is None else x0.copy()
    y, tk = x.copy(), 1.0
    for _ in range(n_iter_max):
        z = y - B.T @ (B @ y - m) / Lip
        x_new = np.sign(z) * np.maximum(np.abs(z) - lam / Lip, 0)
        t_new = (1 + np.sqrt(1 + 4*tk*tk)) / 2
        y = x_new + (tk - 1) / t_new * (x_new - x)
        if np.linalg.norm(x_new - x) <= tol_fista * max(np.linalg.norm(x_new), 1e-30):
            return x_new
        x, tk = x_new, t_new
    return x

def ricostruzione_L1(m_w):
    """L1 con lam per discrepanza (warm start da lam grande)."""
    lam_max = np.abs(B_L1.T @ m_w).max()
    target = N_meas + 2*np.sqrt(2*N_meas)
    x = None
    for f in fattori_L1:
        x = fista_l1(B_L1, m_w, f * lam_max, Lip_L1, x0=x)
        chi2_x = float(np.sum((B_L1 @ x - m_w)**2))
        if chi2_x <= target:
            break
    A_out = np.zeros(N_vox)
    A_out[oss_L1] = x / col_L1[oss_L1]
    return A_out, f, chi2_x

<a id="l1tv"></a>
## 8. Regolarizzazione L1 + variazione totale con ADMM

### 8.1 Variazione totale

La **variazione totale** (TV, rassegna §4.1, eq. 21) penalizza la norma del gradiente dell'immagine:

$$
R_\text{TV}(\mu)=\int|\nabla\mu(\mathbf r)|\,d\mathbf r .
$$

Favorisce immagini **costanti a tratti con bordi netti**: un salto costa quanto la sua ampiezza per l'area del bordo, indipendentemente da quanto è ripido. È la "sparsità del gradiente": invece di pochi voxel accesi, pochi salti.

Sulla griglia di voxel si usa la versione **anisotropa** discreta. Per ogni coppia di voxel adiacenti lungo $x$, $y$ o $z$ (uno **spigolo** $e=(i,k)$ del reticolo) si considera la differenza $x_i-x_k$. Raccogliendo tutte le differenze in una matrice sparsa $D$, con una riga per spigolo ($+1$ sul voxel $i$, $-1$ sul voxel $k$):

$$
(D\mathbf x)_e=x_i-x_k,\qquad \mathrm{TV}(\mathbf x)=\sum_e\big|(D\mathbf x)_e\big|=\|D\mathbf x\|_1 .
$$

Sulla griglia $16\times16\times8$ gli spigoli sono $15\cdot16\cdot8+16\cdot15\cdot8+16\cdot16\cdot7=5632$.

### 8.2 L'ibrido L1+TV

Le due penalità hanno difetti opposti:

- la **L1 da sola** rimpicciolisce l'inclusione a pochi voxel isolati e compensa con un contrasto troppo alto;
- la **TV da sola** produce regioni costanti ma non spinge il fondo a zero, e tende ad allargare le regioni.

Insieme favoriscono un **oggetto piccolo ma compatto e uniforme**, senza fissarne a priori forma o dimensione. In statistica, con TV in 1D, è il *fused lasso* (Tibshirani et al., 2005); in ottica diffusa è stato usato per la tomografia a fluorescenza (Dutta et al., Phys. Med. Biol. 57:1459, 2012). Lo script risolve, con $\mathbf x=\delta\boldsymbol\mu_a$ fisico su **tutti** i 2048 voxel:

$$
\hat{\mathbf x}=\arg\min_{\mathbf x}\;\tfrac12\|W_w\mathbf x-\mathbf m_w\|^2+\lambda\Big[\alpha\sum_jp_j|x_j|+(1-\alpha)\sum_eq_e\,|(D\mathbf x)_e|\Big].
$$

Le quantità:

- $\lambda$ è la forza complessiva della regolarizzazione, scelta con la discrepanza come per la L1;
- $\alpha\in[0,1]$ è il **peso relativo**: $\alpha=1$ è la sola L1, $\alpha=0$ la sola TV;
- $p_j=\max(n_j,\,1/\Delta\mu_\text{max})$ è il peso L1 del voxel: la stessa compensazione di profondità della L1, con un minimo che impedisce ai voxel poco visibili di restare quasi senza penalità (qui non vengono esclusi, perché la TV li lega ai vicini);
- $q_e=\sqrt{p_ip_k}$ è il peso TV dello spigolo $e=(i,k)$: la media geometrica della visibilità dei due voxel.

La TV si applica a $\delta\mu_a$ **fisico**, non alla variabile normalizzata $n_jx_j$: è $\delta\mu_a$ che è costante dentro l'inclusione.

### 8.3 Significato di $\alpha$ e ruolo dei pesi

Si consideri un cubo di lato $L$ voxel con ampiezza $c$, lontano dai bordi, in una zona dove i pesi valgono circa $p$. La sua "massa" è $m\propto cL^3$, ed è ciò che le misure vedono davvero (dimensione e contrasto sono quasi indistinguibili sotto la risoluzione). I due termini valgono:

- L1: $\alpha\,p\,c\,L^3$ ($L^3$ voxel di ampiezza $c$);
- TV: $(1-\alpha)\,p\,c\cdot6L^2$ (le $6L^2$ facce del cubo sono spigoli con salto $c$).

Per unità di massa:

$$
\frac{\text{costo}}{m}\;\propto\;\alpha\,p+(1-\alpha)\,p\,\frac6L .
$$

Il termine L1 non dipende dalla dimensione; quello TV **decresce con $L$**, quindi la TV spinge sempre verso oggetti più grandi e meno intensi, fermata solo dai dati. I due termini si equivalgono alla scala caratteristica

$$
L^*=\frac{6(1-\alpha)}{\alpha}\qquad\Longleftrightarrow\qquad \alpha=\frac{6}{6+L^*} .
$$

Oggetti molto più piccoli di $L^*$ vengono "allargati" dalla TV; per oggetti più grandi decide la L1. Con `alpha_tv_lista = (0.5, 0.7, 0.85)` si ottiene $L^*\approx6$, $2.6$ e $1.1$ voxel (24, 10 e 4 mm).

**Perché i pesi degli spigoli.** Se la TV avesse un peso uniforme mentre la L1 usa $p_j$, in profondità ($p_j$ piccolo) il rapporto TV/L1 crescerebbe come $1/p_j$ e $L^*$ diventerebbe molto più grande: la TV dominerebbe e allargherebbe le inclusioni profonde. Con $q_e=\sqrt{p_ip_k}\approx p$ localmente, il rapporto, e quindi il significato di $\alpha$, è lo stesso a ogni profondità.

### 8.4 Perché non basta FISTA

FISTA funziona per la L1 perché il prossimale di $\|\mathbf x\|_1$ è separabile: si risolve voxel per voxel con la soglia morbida. Il prossimale della penalità L1+TV,

$$
\arg\min_{\mathbf x}\tfrac12\|\mathbf x-\mathbf z\|^2+\tau\Big[\alpha\sum_jp_j|x_j|+(1-\alpha)\sum_eq_e|x_i-x_k|\Big],
$$

**non è separabile**: la TV lega ogni voxel ai suoi vicini, e non esiste una formula chiusa. Serve un algoritmo che "separi" i due termini.

### 8.5 ADMM: derivazione

L'ADMM (*Alternating Direction Method of Multipliers*) introduce variabili ausiliarie per i termini difficili, in modo che ciascun sotto-problema sia facile.

**1. Riformulazione con vincoli.** Si introducono $\mathbf u$ (copia di $\mathbf x$ per la L1) e $\mathbf z$ (copia di $D\mathbf x$ per la TV):

$$
\min_{\mathbf x,\mathbf u,\mathbf z}\;\underbrace{\tfrac12\|W_w\mathbf x-\mathbf m_w\|^2}_{f(\mathbf x)}+\underbrace{\lambda\alpha\sum_jp_j|u_j|}_{h_1(\mathbf u)}+\underbrace{\lambda(1-\alpha)\sum_eq_e|z_e|}_{h_2(\mathbf z)}\quad\text{con}\quad \mathbf x-\mathbf u=0,\ \ D\mathbf x-\mathbf z=0 .
$$

Il problema è identico all'originale, ma ora la parte non liscia dipende da variabili separate, e $h_1$, $h_2$ sono **separabili** componente per componente.

**2. Lagrangiana aumentata.** Si aggiungono moltiplicatori $\mathbf y_u$, $\mathbf y_z$ e penalità quadratiche sui vincoli, con matrici diagonali positive $P_u=\operatorname{diag}(\rho_{u,j})$ e $P_z=\operatorname{diag}(\rho_{z,e})$:

$$
\mathcal L=f(\mathbf x)+h_1(\mathbf u)+h_2(\mathbf z)
+\mathbf y_u^T(\mathbf x-\mathbf u)+\tfrac12(\mathbf x-\mathbf u)^TP_u(\mathbf x-\mathbf u)
+\mathbf y_z^T(D\mathbf x-\mathbf z)+\tfrac12(D\mathbf x-\mathbf z)^TP_z(D\mathbf x-\mathbf z).
$$

I termini lineari impongono i vincoli (come in un normale Lagrangiano); quelli quadratici li rendono "elastici" e stabilizzano l'algoritmo.

**3. Forma scalata.** Con i moltiplicatori scalati $\mathbf a=P_u^{-1}\mathbf y_u$ e $\mathbf b=P_z^{-1}\mathbf y_z$, completando il quadrato,

$$
\mathbf y_u^T\mathbf v+\tfrac12\mathbf v^TP_u\mathbf v=\tfrac12(\mathbf v+\mathbf a)^TP_u(\mathbf v+\mathbf a)-\tfrac12\mathbf a^TP_u\mathbf a ,
$$

e lo stesso per $\mathbf z$. La Lagrangiana diventa, a meno di termini che non dipendono da $\mathbf x$, $\mathbf u$, $\mathbf z$:

$$
\mathcal L=f(\mathbf x)+h_1(\mathbf u)+h_2(\mathbf z)+\tfrac12\|\mathbf x-\mathbf u+\mathbf a\|_{P_u}^2+\tfrac12\|D\mathbf x-\mathbf z+\mathbf b\|_{P_z}^2,\qquad \|\mathbf v\|_P^2=\mathbf v^TP\mathbf v .
$$

**4. Minimizzazioni alternate.** L'ADMM minimizza $\mathcal L$ rispetto a una variabile alla volta, poi aggiorna i moltiplicatori.

*Passo in $\mathbf x$.* $\mathcal L$ è quadratica in $\mathbf x$; ponendo a zero il gradiente,

$$
W_w^T(W_w\mathbf x-\mathbf m_w)+P_u(\mathbf x-\mathbf u+\mathbf a)+D^TP_z(D\mathbf x-\mathbf z+\mathbf b)=0
$$

$$
\Longrightarrow\quad \underbrace{\big(W_w^TW_w+P_u+D^TP_zD\big)}_{K}\,\mathbf x=W_w^T\mathbf m_w+P_u(\mathbf u-\mathbf a)+D^TP_z(\mathbf z-\mathbf b).
$$

È un Tikhonov generalizzato: fedeltà ai dati più due termini che tirano $\mathbf x$ verso $\mathbf u-\mathbf a$ e $D\mathbf x$ verso $\mathbf z-\mathbf b$.

*Passo in $\mathbf u$.* Restano solo $h_1$ e il quadrato in $\mathbf u$, separabili:

$$
\min_{u_j}\;\lambda\alpha p_j|u_j|+\tfrac{\rho_{u,j}}2\big(u_j-(x_j+a_j)\big)^2 .
$$

Dividendo per $\rho_{u,j}$ è esattamente l'esempio scalare del §7.5, quindi

$$
u_j=\mathcal S_{\lambda\alpha p_j/\rho_{u,j}}\big(x_j+a_j\big).
$$

*Passo in $\mathbf z$.* Allo stesso modo, spigolo per spigolo:

$$
z_e=\mathcal S_{\lambda(1-\alpha)q_e/\rho_{z,e}}\big((D\mathbf x)_e+b_e\big).
$$

Questa soglia **azzera i salti piccoli** fra voxel vicini: è il meccanismo con cui la TV crea regioni costanti.

*Aggiornamento dei moltiplicatori* (salita duale, $\mathbf y_u\leftarrow\mathbf y_u+P_u(\mathbf x-\mathbf u)$, cioè in forma scalata):

$$
\mathbf a\leftarrow\mathbf a+\mathbf x-\mathbf u,\qquad \mathbf b\leftarrow\mathbf b+D\mathbf x-\mathbf z .
$$

**Intuizione.** $\mathbf a$ e $\mathbf b$ **accumulano** la violazione dei vincoli, come il termine integrale di un controllore: finché $\mathbf x$ e $\mathbf u$ non coincidono, la differenza si somma in $\mathbf a$ e spinge il passo successivo a ridurla. A convergenza $\mathbf x=\mathbf u$ e $D\mathbf x=\mathbf z$, e $\mathbf u$ è il minimo del problema originale.

### 8.6 Scelte numeriche

**Penalità diagonali.** Per un problema convesso l'ADMM converge con qualsiasi $\rho>0$, ma la velocità dipende molto dalla scala. La diagonale di $W_w^TW_w$ vale $n_j^2$, che varia di ordini di grandezza fra superficie e profondità; con un $\rho$ unico alcuni voxel convergerebbero subito e altri lentissimamente. Lo script usa

$$
\rho_{u,j}=c\,p_j^2,\qquad \rho_{z,e}=c\,q_e^2\qquad(c=\texttt{rho\_rel\_tv}),
$$

così ogni voxel e ogni spigolo hanno una penalità proporzionale alla propria curvatura. La soglia del passo in $\mathbf u$ diventa $\lambda\alpha/(c\,p_j)$: lo stesso andamento $1/n_j$ della soglia della L1 (§7.6).

**Matrice precalcolata.** $K=W_w^TW_w+P_u+D^TP_zD$ non dipende né da $\lambda$ né da $\alpha$ (i $\rho$ dipendono solo dai pesi). Si inverte quindi **una volta per lunghezza d'onda** (`Minv_tv` in `prepara_pesi`), e ogni iterazione costa un prodotto matrice-vettore $2048\times2048$. $K$ è definita positiva grazie a $P_u$, quindi l'inversa esiste sempre.

**Arresto.** Ci si ferma quando il residuo primale (quanto i vincoli sono violati) e la variazione di $\mathbf u$ sono piccoli rispetto a $\|\mathbf x\|$:

$$
\sqrt{\|\mathbf x-\mathbf u\|^2+\|D\mathbf x-\mathbf z\|^2}\le\varepsilon\|\mathbf x\|,\qquad \|\mathbf u-\mathbf u_\text{prec}\|\le\varepsilon\|\mathbf x\| ,
$$

con $\varepsilon$ = `tol_admm`. La variazione di $\mathbf u$ è un indicatore semplice del residuo duale. Si restituisce $\mathbf u$, che ha zeri esatti.

### 8.7 $\lambda_\text{max}$ per la L1+TV

In $\mathbf x=0$ la condizione di ottimalità è

$$
0\in-W_w^T\mathbf m_w+\lambda\alpha\,P\,\partial|\mathbf 0|+\lambda(1-\alpha)\,D^TQ\,\partial|\mathbf 0| ,
$$

con $P=\operatorname{diag}(p_j)$ e $Q=\operatorname{diag}(q_e)$. Scegliendo il subgradiente della TV uguale a zero, basta che $|(W_w^T\mathbf m_w)_j|\le\lambda\alpha p_j$ per ogni $j$. Quindi

$$
\lambda\ge\max_j\frac{|(W_w^T\mathbf m_w)_j|}{\alpha\,p_j}
$$

garantisce l'immagine vuota. È una **maggiorazione**: il vero valore di soglia può essere più basso, perché anche i subgradienti della TV possono contribuire. Per questo il primo valore della griglia può già dare un'immagine vuota, e la discrepanza prosegue con i successivi. La griglia `fattori_L1` e il warm start sono gli stessi della L1; tra un $\lambda$ e il successivo si conservano $\mathbf u$, $\mathbf z$, $\mathbf a$, $\mathbf b$.

### Implementazione: parametri

- `alpha_tv_lista`: i valori di $\alpha$ da confrontare. Ognuno diventa un metodo distinto (`L1_TV_0.5`, ...) con la sua colonna `alpha_tv` nell'Excel, e aggiunge una ricostruzione per caso.
- `rho_rel_tv` è la costante $c$ delle penalità diagonali.
- `tol_admm` e `n_iter_admm` controllano l'arresto per ogni valore di $\lambda$.

In [ ]:
# --- L1 + TV (ADMM) ------------------------------------------------------------
# Variabili ausiliarie u = x (L1) e z = D x (TV), moltiplicatori scalati a, b.
# Pesi: p_j per i voxel, q_e = sqrt(p_i p_k) per lo spigolo e = (i,k).
# Penalita' dell'ADMM DIAGONALI (rho_u,j = c p_j^2, rho_z,e = c q_e^2): le
# sensibilita' differiscono di ordini di grandezza fra superficie e profondita'
# e un rho unico rallenterebbe molto la convergenza.
#   x <- (W_w^T W_w + diag(rho_u) + D^T diag(rho_z) D)^-1
#            (W_w^T m_w + rho_u (u - a) + D^T (rho_z (z - b)))
#   u <- S_{lam alpha p_j / rho_u,j}(x + a)          (soft-thresholding dei voxel)
#   z <- S_{lam (1-alpha) q_e / rho_z,e}(D x + b)    (soft-thresholding dei salti)
#   a <- a + x - u,   b <- b + D x - z
# La matrice del primo passo non dipende ne' da lam ne' da alpha: la si
# inverte una volta per lambda (in prepara_pesi), cosi' ogni iterazione costa
# un prodotto matrice-vettore. La soluzione restituita e' u (zeri esatti).
# alpha_tv_lista: valori di alpha da confrontare (1 = solo L1, 0 = solo TV).
#   L* = 6(1-alpha)/alpha voxel: 0.5 -> 6 voxel (24 mm), 0.7 -> 2.6 voxel
#   (~10 mm), 0.85 -> 1.1 voxel (~4 mm). Ogni valore aggiunge una ricostruzione
#   per caso: per scan piu' rapidi lasciarne uno solo.
alpha_tv_lista = (0.5, 0.7, 0.85)
rho_rel_tv = 1.0      # c: rho_u = c p_j^2, rho_z = c q_e^2
tol_admm = 1e-3       # arresto: residuo primale e variazione di u relativi < tol_admm
n_iter_admm = 500     # iterazioni massime per ogni valore di lam

### `matrice_differenze`: costruzione di $D$

- `idx3 = np.arange(N_vox).reshape(Nx, Ny, Nz)` mette l'indice lineare di ogni voxel nella sua posizione 3D, con lo stesso ordinamento di `r_V`.
- `idx3[1:, :, :]` e `idx3[:-1, :, :]` sono i voxel "successivo" e "precedente" lungo $x$: lo slicing sfasato di uno produce tutte le coppie adiacenti senza cicli. Lo stesso per $y$ e $z$. `ravel()` appiattisce, `np.concatenate` unisce le tre direzioni.
- `sparse.csr_matrix((dati, (righe, colonne)), shape=...)` costruisce la matrice a partire dalle triplette: per lo spigolo $e$ un $+1$ nella colonna del voxel `piu` e un $-1$ in quella del voxel `meno`. Il formato CSR rende veloci i prodotti `D_tv @ x` e `D_tv.T @ v`.
- La funzione restituisce anche gli indici `piu` e `meno`, che servono a calcolare i pesi degli spigoli $q_e=\sqrt{p_ip_k}$ in `prepara_pesi`.

### `soft`

La soglia morbida $\mathcal S_\tau$ con soglia **vettoriale**: `soglia` può essere un array della stessa lunghezza di `v`, perché nei passi in $\mathbf u$ e $\mathbf z$ ogni voxel e ogni spigolo hanno la propria soglia.

In [ ]:
def matrice_differenze():
    """D (sparsa): una riga per ogni coppia di voxel adiacenti lungo x, y, z,
    piu' gli indici (piu, meno) dei due voxel di ogni spigolo."""
    idx3 = np.arange(N_vox).reshape(Nx, Ny, Nz)
    piu = np.concatenate([idx3[1:, :, :].ravel(), idx3[:, 1:, :].ravel(), idx3[:, :, 1:].ravel()])
    meno = np.concatenate([idx3[:-1, :, :].ravel(), idx3[:, :-1, :].ravel(), idx3[:, :, :-1].ravel()])
    n_e = piu.size
    righe = np.concatenate([np.arange(n_e), np.arange(n_e)])
    D = sparse.csr_matrix((np.concatenate([np.ones(n_e), -np.ones(n_e)]),
                           (righe, np.concatenate([piu, meno]))), shape=(n_e, N_vox))
    return D, piu, meno

D_tv, spigolo_piu, spigolo_meno = matrice_differenze()

def soft(v, soglia):
    return np.sign(v) * np.maximum(np.abs(v) - soglia, 0.0)

### `ricostruzione_L1TV`: l'ADMM, riga per riga

| Riga di codice | Formula |
|---|---|
| `Wtm = W_w.T @ m_w` | $W_w^T\mathbf m_w$, calcolato una volta |
| `lam_max = np.max(np.abs(Wtm) / (alpha * peso_tv))` | maggiorazione di $\lambda_\text{max}$ (§8.7) |
| `u, a, z, b = 0` | variabili ausiliarie e moltiplicatori scalati, inizializzati a zero e poi conservati fra un $\lambda$ e l'altro |
| `s1 = lam * alpha * peso_tv / rho_u_tv` | soglie dei voxel $\lambda\alpha p_j/\rho_{u,j}$ |
| `s2 = lam * (1 - alpha) * peso_spigoli_tv / rho_z_tv` | soglie degli spigoli $\lambda(1-\alpha)q_e/\rho_{z,e}$ |
| `x = Minv_tv @ (Wtm + rho_u_tv * (u - a) + D_tv.T @ (rho_z_tv * (z - b)))` | passo in $\mathbf x$: $K^{-1}\big[W_w^T\mathbf m_w+P_u(\mathbf u-\mathbf a)+D^TP_z(\mathbf z-\mathbf b)\big]$; il prodotto per una diagonale è un prodotto elemento per elemento |
| `u = soft(x + a, s1)` | passo in $\mathbf u$ |
| `z = soft(Dx + b, s2)` | passo in $\mathbf z$ |
| `a += x - u`, `b += Dx - z` | aggiornamento dei moltiplicatori (in place) |
| `if (... <= tol_admm * scala and ...)` | criterio d'arresto del §8.6 |
| `chi2_u = ...; if chi2_u <= target: break` | discrepanza, come per la L1 |

La funzione restituisce $\mathbf u$ direttamente in $\delta\mu_a$ (mm⁻¹): a differenza della L1, qui non c'è cambio di variabile.

In [ ]:
def ricostruzione_L1TV(m_w, alpha):
    """L1+TV con peso relativo alpha e lam per discrepanza (warm start da lam grande)."""
    Wtm = W_w.T @ m_w
    lam_max = np.max(np.abs(Wtm) / (alpha * peso_tv))     # per lam >= lam_max, x = 0 e' ottimo
    target = N_meas + 2*np.sqrt(2*N_meas)
    u = np.zeros(N_vox); a = np.zeros(N_vox)
    z = np.zeros(D_tv.shape[0]); b = np.zeros(D_tv.shape[0])
    for f in fattori_L1:
        lam = f * lam_max
        s1 = lam * alpha * peso_tv / rho_u_tv
        s2 = lam * (1 - alpha) * peso_spigoli_tv / rho_z_tv
        for _ in range(n_iter_admm):
            x = Minv_tv @ (Wtm + rho_u_tv * (u - a) + D_tv.T @ (rho_z_tv * (z - b)))
            Dx = D_tv @ x
            u_old = u
            u = soft(x + a, s1)
            z = soft(Dx + b, s2)
            a += x - u
            b += Dx - z
            scala = max(np.linalg.norm(x), 1e-30)
            if (np.sqrt(np.sum((x - u)**2) + np.sum((Dx - z)**2)) <= tol_admm * scala
                    and np.linalg.norm(u - u_old) <= tol_admm * scala):
                break
        chi2_u = float(np.sum((W_w @ u - m_w)**2))
        if chi2_u <= target:
            break
    return u, f, chi2_u

<a id="metriche"></a>
## 9. Soglie di significatività, componenti connesse e metriche

### 9.1 Segnali in unità di deviazione standard

Le misure sbiancate hanno rumore a varianza 1. Un contributo ricostruito $a\,\mathbf w$ (un'ampiezza $a$ per un'impronta $\mathbf w$, colonna o somma di colonne di $W_w$) ha norma $|a|\,\|\mathbf w\|$: è il suo **segnale in deviazioni standard**. Lo script usa questa grandezza per due decisioni:

- un voxel della L1 è **acceso** se $n_j|\hat A_j|\ge2$ (`soglia_sigma_voxel`) e se supera l'1% del massimo (`soglia_acceso`, che scarta i residui numerici);
- un'inclusione è **trovata** se il segnale della regione scelta supera `soglia_trovata[metodo]`.

### 9.2 Effetto *look-elsewhere* e calibrazione delle soglie

Ogni metodo sceglie il **massimo** fra molti candidati (il picco fra 2048 voxel, la componente più forte). Anche con solo rumore il massimo di molte variabili è grande: alcune deviazioni standard, in misura diversa per ogni metodo. Una soglia fissa a $3\sigma$ darebbe quindi tassi di falsi positivi diversi e non confrontabili.

Prima dello scan si simulano misure di **solo rumore** ($\delta\mu_a=0$), si ricostruiscono con ogni metodo e si prende come soglia il quantile 0.99 del segnale trovato: così un'inclusione inesistente viene dichiarata "trovata" nell'1% dei casi, per tutti i metodi. Due accorgimenti: nessuna soglia L1 scende sotto `soglia_sigma_voxel`, e quella della L1+TV non scende sotto quella della L1 (sul solo rumore la L1+TV resta quasi sempre vuota, e la calibrazione da sola le darebbe una soglia troppo permissiva).

### Implementazione

`METODI_TV = {f"L1_TV_{a:g}": a for a in alpha_tv_lista}` è una *dict comprehension* che crea un nome di metodo per ogni $\alpha$; `METODI` concatena le tuple dei nomi. `soglia_trovata` parte da 3σ e viene sovrascritto dalla calibrazione.

In [ ]:
# --- soglie di significativita' (in unita' di deviazioni standard del rumore) --
# I dati pesati m_w = M/sigma hanno rumore unitario: un contributo ricostruito
# il cui segnale ha norma |dmu| * ||w|| (w = colonna o somma di colonne di W_w)
# vale "quante sigma" sopra il rumore. Si usa per entrambe le decisioni:
#   - voxel L1 acceso:      ||w_j|| * |A_j|           >= soglia_sigma_voxel
#   - inclusione trovata:   ||w_C|| * |dmu_fit,C|     >= soglia_trovata[metodo]
#     (C = cluster o componente scelta)
# Nessun metodo usa la posizione vera per scegliere cluster o componente.
# SOGLIA DI "TROVATA" CALIBRATA SUL RUMORE (per metodo): ogni metodo sceglie il
# massimo fra molti candidati (picco fra 2048 voxel, componente piu' forte),
# quindi anche il solo rumore produce "segnali" di alcune sigma, in misura
# diversa per metodo (effetto look-elsewhere: TSVD ~1-2, Tikhonov ~3-5, L1 ~4-8).
# Prima dello scan si simulano misure di SOLO rumore (dmu_a = 0) e la soglia di
# ciascun metodo e' il quantile_falsi_positivi del segnale trovato: con 0.99,
# un'inclusione inesistente viene dichiarata "trovata" nell'1% dei casi.
# La calibrazione usa lo STESSO modello_sigma dello scan.
soglia_acceso = 0.01        # L1: scarta anche i residui numerici < 1% del massimo
soglia_sigma_voxel = 2.0    # L1: segnale del singolo voxel >= 2 sigma
quantile_falsi_positivi = 0.99
lambda_calibrazione = (650.0, 800.0, 950.0)   # [nm] lambda usate per la calibrazione
n_realizzazioni_rumore = 25                   # misure di solo rumore per lambda
METODI_TV = {f"L1_TV_{a:g}": a for a in alpha_tv_lista}      # nome del metodo -> alpha
METODI = ("TSVD", "Tikhonov", "L1") + tuple(METODI_TV)
soglia_trovata = {m: 3.0 for m in METODI}   # sovrascritte dalla calibrazione

### 9.3 Componenti connesse

I voxel accesi vanno raggruppati in **regioni contigue**: ogni regione è un candidato inclusione. Due voxel sono vicini se differiscono di al più 1 in ciascun indice (26-vicinato in 3D).

**Implementazione** (`componenti_connesse`): è un algoritmo di *flood fill*.

1. Si scorrono i voxel attivi (`zip(*np.nonzero(maschera_3d))` restituisce le terne di indici).
2. Se un voxel non ha ancora un'etichetta, si apre una nuova componente e lo si mette in una lista usata come **pila** (`coda.pop()` estrae l'ultimo elemento: visita in profondità).
3. Si estraggono voxel dalla pila e si aggiungono i vicini attivi non ancora etichettati, finché la pila si svuota.

`componenti_con_segno` ripete la procedura separatamente per i voxel positivi e negativi, così una zona positiva e una negativa adiacenti restano due componenti distinte; le etichette delle negative sono spostate di `n_out` per non sovrapporsi.

In [ ]:
# --- componenti connesse (26-vicinato), separate per segno -------------------
def componenti_connesse(maschera_3d):
    etichette = np.zeros(maschera_3d.shape, int)
    n_comp = 0
    for start in zip(*np.nonzero(maschera_3d)):
        if etichette[start]:
            continue
        n_comp += 1
        etichette[start] = n_comp
        coda = [start]
        while coda:
            i, j, k = coda.pop()
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    for dk in (-1, 0, 1):
                        q = (i + di, j + dj, k + dk)
                        if (0 <= q[0] < Nx and 0 <= q[1] < Ny and 0 <= q[2] < Nz
                                and maschera_3d[q] and not etichette[q]):
                            etichette[q] = n_comp
                            coda.append(q)
    return etichette.flatten(), n_comp

def componenti_con_segno(A_vec, attivi):
    """Componenti connesse dei voxel 'attivi' (maschera booleana), separate
    per segno di A_vec (prima le positive, poi le negative)."""
    etich_out = np.zeros(N_vox, int)
    n_out = 0
    for segno in (+1, -1):
        e, nc = componenti_connesse((attivi & (segno * A_vec > 0)).reshape(Nx, Ny, Nz))
        etich_out[e > 0] = e[e > 0] + n_out
        n_out += nc
    return etich_out, n_out

### 9.4 Metriche e stima di $\delta\mu_a$

Nessuna metrica usa la posizione vera per scegliere la regione: la posizione vera serve solo a calcolare l'errore.

**`metriche_cluster` (TSVD, Tikhonov).** Si prende il voxel di picco di $|\hat{\mathbf A}|$ e il **cluster** $\mathcal C$ dei voxel dello stesso segno con almeno il 50% del picco. Poi si stima $\delta\mu_a$ supponendolo **costante sul cluster** e rifittando i dati:

$$
\min_a\big\|a\,\mathbf w_\mathcal C-\mathbf m_w\big\|^2,\qquad \mathbf w_\mathcal C=\sum_{j\in\mathcal C}\mathbf w_j
\quad\Longrightarrow\quad
\widehat{\delta\mu}_\mathcal C=\frac{\mathbf w_\mathcal C^T\mathbf m_w}{\mathbf w_\mathcal C^T\mathbf w_\mathcal C},
$$

ottenuto ponendo a zero la derivata $2\mathbf w_\mathcal C^T(a\mathbf w_\mathcal C-\mathbf m_w)$. Il segnale è $|\widehat{\delta\mu}_\mathcal C|\,\|\mathbf w_\mathcal C\|$. Questo è un **hard prior** con una sola regione (rassegna §4.2): la colonna della regione è la somma delle colonne dei suoi voxel. La regione però viene dalla ricostruzione stessa, non da un'altra modalità di imaging.

**`metriche_L1` (L1, L1+TV).** I voxel accesi si dividono in componenti $\mathcal C_1,\dots,\mathcal C_K$ e si fa un fit **congiunto senza penalità**, un'ampiezza per componente:

$$
\hat{\mathbf a}=\arg\min_{\mathbf a}\Big\|\sum_ka_k\,\boldsymbol\omega_k-\mathbf m_w\Big\|^2,\qquad \boldsymbol\omega_k=\sum_{j\in\mathcal C_k}\mathbf w_j .
$$

È il **debiasing**: la L1 serve solo a trovare il supporto, e le ampiezze, che la soglia morbida avrebbe ridotto, sono stimate dai minimi quadrati. Si sceglie la componente con il segnale più forte, $\max_k|\hat a_k|\,\|\boldsymbol\omega_k\|$.

**`esito`** raccoglie le metriche: baricentro pesato con $|\hat A_j|$, $\bar{\mathbf r}=\sum_j|\hat A_j|\mathbf r_j/\sum_j|\hat A_j|$, errore di localizzazione, voxel in comune con l'inclusione vera, rapporto fra $\delta\mu_a$ stimato e vero. Se il segnale è sotto soglia, posizione ed errore sono `NaN` e i valori restano nelle colonne `*_grezzo`.

### Implementazione

- `w[:, None] * r_V[sel]` moltiplica ogni riga di coordinate per il suo peso (broadcasting) e `.sum(0)` somma sui voxel: è il baricentro in una riga.
- `np.column_stack([... for c in range(1, n_comp + 1)])` costruisce la matrice $[\boldsymbol\omega_1\cdots\boldsymbol\omega_K]$; `np.linalg.lstsq` risolve i minimi quadrati, e `a_comp, *_ = ...` tiene solo la soluzione scartando gli altri valori restituiti.
- `classe_snr` assegna una classe di SNR atteso, usata come filtro nei pivot.

In [ ]:
# --- metriche ----------------------------------------------------------------
soglia_cluster = 0.5      # TSVD/Tikhonov: cluster = voxel >= 50% del picco (stesso segno)

def esito(A_hat, sel, fit, segnale, soglia, mask, r_true, n_comp=np.nan, err_vicina=np.nan):
    """Dizionario delle metriche per l'insieme di voxel 'sel' (cluster o
    componente). Se il segnale e' sotto soglia l'inclusione e' 'non trovata':
    posizione, errore e dmu_fit sono vuoti (restano nelle colonne *_grezzo)."""
    w = np.abs(A_hat[sel])
    r_bar = (w[:, None] * r_V[sel]).sum(0) / w.sum()
    jj = np.where(sel)[0][np.argmax(w)]
    err = float(np.linalg.norm(r_bar - r_true))
    trovata = bool(segnale >= soglia)
    nan = np.nan
    return dict(trovata=trovata, segnale_sigma=float(segnale),
                x_rec=r_bar[0] if trovata else nan, y_rec=r_bar[1] if trovata else nan,
                z_rec=r_bar[2] if trovata else nan,
                errore_baricentro_mm=err if trovata else nan, errore_baricentro_grezzo_mm=err,
                errore_picco_mm=float(np.linalg.norm(r_V[jj] - r_true)) if trovata else nan,
                dmu_fit=fit if trovata else nan, dmu_fit_grezzo=fit,
                dmu_picco=float(A_hat[jj]),
                voxel_ricostruiti=int(sel.sum()), voxel_in_comune=int((sel & mask).sum()),
                n_componenti=n_comp, errore_comp_piu_vicina_mm=err_vicina)

def metriche_cluster(A_hat, m_w, mask, r_true, dmu_vero, soglia=np.inf):
    """TSVD/Tikhonov: picco di |A_hat|, cluster al 50% del picco, baricentro
    pesato con |A_hat| e fit pesato di dmu_a costante sul cluster."""
    j = int(np.argmax(np.abs(A_hat)))
    sgn = np.sign(A_hat[j]) if A_hat[j] != 0 else 1.0
    cl = sgn * A_hat >= soglia_cluster * np.abs(A_hat[j])
    w_c = W_w[:, cl].sum(axis=1)
    fit = float((w_c @ m_w) / (w_c @ w_c))
    return esito(A_hat, cl, fit, abs(fit) * np.linalg.norm(w_c), soglia, mask, r_true)

def metriche_L1(A_hat, m_w, mask, r_true, dmu_vero, soglia=np.inf):
    """L1: voxel accesi = sopra l'1% del massimo E con segnale >= 2 sigma;
    componenti connesse per segno; fit congiunto senza penalita'. Si sceglie la
    componente con il SEGNALE PIU' FORTE (||a_k w_k|| massimo), senza usare la
    posizione vera. 'errore_comp_piu_vicina_mm' riporta, solo per confronto,
    la distanza della componente piu' vicina al vero (metrica 'oracolo' delle
    versioni precedenti, ottimistica quando ci sono molte componenti di rumore)."""
    a = np.abs(A_hat)
    accesi = (a > soglia_acceso * a.max()) & (col_L1 * a >= soglia_sigma_voxel) if a.max() > 0 \
        else np.zeros(N_vox, bool)
    etich, n_comp = componenti_con_segno(A_hat, accesi)
    if n_comp == 0:
        nan = np.nan
        return dict(trovata=False, segnale_sigma=0.0, x_rec=nan, y_rec=nan, z_rec=nan,
                    errore_baricentro_mm=nan, errore_baricentro_grezzo_mm=nan,
                    errore_picco_mm=nan, dmu_fit=nan, dmu_fit_grezzo=nan, dmu_picco=nan,
                    voxel_ricostruiti=0, voxel_in_comune=0, n_componenti=0,
                    errore_comp_piu_vicina_mm=nan)
    W_comp = np.column_stack([W_w[:, etich == c].sum(axis=1) for c in range(1, n_comp + 1)])
    a_comp, *_ = np.linalg.lstsq(W_comp, m_w, rcond=None)
    segnali = np.abs(a_comp) * np.linalg.norm(W_comp, axis=0)     # ||a_k w_k||
    k = int(np.argmax(segnali))
    dist = []
    for c in range(1, n_comp + 1):
        sel = etich == c
        w = a[sel]
        dist.append(np.linalg.norm((w[:, None] * r_V[sel]).sum(0) / w.sum() - r_true))
    return esito(A_hat, etich == k + 1, float(a_comp[k]), float(segnali[k]), soglia, mask, r_true,
                 n_comp=int(n_comp), err_vicina=float(min(dist)))


def classe_snr(snr):
    """Classe del SNR atteso, utile come filtro nei pivot: sotto ~5 l'inclusione
    e' sotto il rumore e i rapporti dmu_a stimato/vero non hanno significato."""
    for lim, nome in [(5, "1: <5 (sotto il rumore)"), (20, "2: 5-20 (limite)"),
                      (100, "3: 20-100"), (np.inf, "4: >100")]:
        if snr < lim:
            return nome

<a id="excel"></a>
## 10. Salvataggio in Excel

I risultati sono una lista di dizionari, uno per caso e metodo, trasformata in `DataFrame`. Il file contiene:

- **`dati`**: formato "lungo", una riga per caso × metodo. È il formato ideale per tabelle e grafici pivot (filtri per `metodo`, `alpha_tv`, `classe_SNR`, ...).
- **`confronto`**: formato "largo", una riga per caso con i metodi affiancati, ottenuto con `pivot_table(index=chiavi, columns="metodo", aggfunc="first")`; i nomi delle colonne a due livelli sono appiattiti in `variabile_metodo`.
- **`spettri`**, **`posizioni`**, **`variazioni`**, **`configurazione`**: i parametri della simulazione, per rendere il file autosufficiente.

`pd.ExcelWriter(..., engine="openpyxl")` scrive tutti i fogli; `aggiungi_grafici_spettri` usa `openpyxl.chart` per inserire grafici Excel nativi di $\mu_a$, $\mu_s'$ e $\delta\mu_a(\lambda)$, che si aggiornano se si modificano i dati del foglio.

In [ ]:
# %% =============================================================
# STEP 12 (funzione) - Salvataggio Excel
# ==================================================================
def salva_excel(righe, percorso):
    """Fogli: 'dati' (formato lungo, una riga per caso e metodo: ideale per i
    pivot), 'confronto' (una riga per caso, metodi affiancati), 'spettri',
    'posizioni', 'variazioni', 'configurazione'."""
    df = pd.DataFrame(righe)
    chiavi = ["lambda_nm", "pos_id", "var_id"]
    wide = df.pivot_table(index=chiavi, columns="metodo",
                          values=["trovata", "errore_baricentro_mm", "rapporto_fit", "dmu_fit",
                                  "voxel_in_comune", "segno_corretto"],
                          aggfunc="first")
    wide.columns = [f"{v}_{m}" for v, m in wide.columns]
    info = df.drop_duplicates(chiavi)[chiavi + ["pos_desc", "x_vero", "y_vero", "z_vero",
                                                "var_desc", "d_lip", "d_HHb", "d_O2Hb", "d_tHb",
                                                "SO2_incl", "mu_a0",
                                                "mu_s0", "dmu_a_vero", "dmu_su_mua0",
                                                "SNR_atteso", "classe_SNR",
                                                "sigma_dati_su_sigma0"]]
    wide = info.merge(wide.reset_index(), on=chiavi)
    spettri_df = pd.DataFrame({"lambda_nm": wl_nm, "mu_a_bkg [mm^-1]": mu_a_bkg,
                               "mu_s'_bkg [mm^-1]": mu_s_bkg,
                               **{f"dmu_a {v[0]} {v[1]} [mm^-1]": dmu_var[v[0]] for v in variazioni}})
    pos_df = pd.DataFrame([{"pos_id": p[0], "descrizione": p[1], "x": p[2][0], "y": p[2][1],
                            "z": p[2][2]} for p in posizioni])
    var_df = pd.DataFrame([{"var_id": v[0], "descrizione": v[1], "d_lip": v[2],
                            "d_HHb": v[3], "d_O2Hb": v[4], "d_tHb": v[3] + v[4],
                            **{f"incl_{k}": comp_var[v[0]][k]
                               for k in ("HHb", "O2Hb", "tHb", "SO2", "Lipid", "H2O")}}
                           for v in variazioni])
    testo_pesi = ("sigma stimata dai conteggi osservati di ogni misura: "
                  "sqrt((n_pert+1)/(n0+1)^2 + (n_pert+1)^2/(n0+1)^3); SVD per caso"
                  if modello_sigma == "dati" else
                  "sigma_0 = sqrt(2/(N0+1)) dai conteggi attesi di baseline; SVD per lambda")
    conf_df = pd.DataFrame({"parametro": ["composizione background", "scattering",
                                          "N_tot_TPSF", "coppie", "gate", "voxel",
                                          "dmu_a_max (osservabilita' L1)", "soglia_cluster",
                                          "soglia_acceso", "seme_rumore",
                                          "modello_sigma", "pesi",
                                          "soglia_sigma_voxel (L1)", "soglie 'trovata' (calibrate)",
                                          "scelta cluster/componente", "L1+TV"],
                            "valore": [str(composizione),
                                       f"Jacques fatty: a={a_jac} mm^-1, b={b_jac}, l0={wl0_jac} nm",
                                       N_tot_TPSF, f"{N_pairs} ({modalita_coppie})",
                                       f"{N_gates} x {gate_width} ns da {t_start} ns",
                                       f"{Nx}x{Ny}x{Nz}, lato {step} mm", dmu_a_max,
                                       soglia_cluster, soglia_acceso, seme_rumore,
                                       modello_sigma, testo_pesi,
                                       f"{soglia_sigma_voxel}: voxel acceso se ||w_j||*|A_j| >= soglia",
                                       "trovata se ||w_C||*|dmu_fit| >= soglia: " + ", ".join(
                                           f"{m} {s:.2f}" for m, s in soglia_trovata.items())
                                       + f" sigma (quantile {quantile_falsi_positivi} del solo rumore)",
                                       "TSVD/Tik: cluster del picco; L1 e L1+TV: componente con segnale "
                                       "massimo (nessuna informazione sulla posizione vera)",
                                       f"ADMM, alpha in {alpha_tv_lista} (L1 pesata con p_j = max(n_j, "
                                       f"1/dmu_a_max), TV anisotropa pesata con sqrt(p_i p_k)), "
                                       f"c={rho_rel_tv}, tol={tol_admm}, max {n_iter_admm} iterazioni per lam"]})
    with pd.ExcelWriter(percorso, engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name="dati", index=False)
        wide.to_excel(xw, sheet_name="confronto", index=False)
        spettri_df.to_excel(xw, sheet_name="spettri", index=False)
        pos_df.to_excel(xw, sheet_name="posizioni", index=False)
        var_df.to_excel(xw, sheet_name="variazioni", index=False)
        conf_df.to_excel(xw, sheet_name="configurazione", index=False)
        aggiungi_grafici_spettri(xw.book["spettri"], n_righe=len(spettri_df),
                                 n_var=len(variazioni))
    return df


def aggiungi_grafici_spettri(ws, n_righe, n_var):
    """Grafici Excel nativi nel foglio 'spettri': mu_a e mu_s' di background e
    dmu_a di ogni variazione in funzione di lambda (si aggiornano se si
    modificano i dati del foglio)."""
    from openpyxl.chart import ScatterChart, Reference, Series
    x = Reference(ws, min_col=1, min_row=2, max_row=n_righe + 1)
    def grafico(titolo, y_label, colonne, ancora):
        ch = ScatterChart()
        ch.title, ch.style = titolo, 13
        ch.x_axis.title, ch.y_axis.title = "lambda [nm]", y_label
        ch.x_axis.scaling.min, ch.x_axis.scaling.max = float(wl_nm[0]), float(wl_nm[-1])
        ch.x_axis.delete = False; ch.y_axis.delete = False
        ch.x_axis.tickLblPos = "low"               # etichette di lambda in basso anche se y < 0
        ch.height, ch.width = 8, 16
        for c in colonne:
            y = Reference(ws, min_col=c, min_row=1, max_row=n_righe + 1)
            s = Series(y, x, title_from_data=True)
            s.marker.symbol = "circle"; s.marker.size = 4
            s.smooth = False                       # segmenti fra i punti, nessuna interpolazione
            ch.series.append(s)
        ws.add_chart(ch, ancora)
    grafico("Assorbimento di background", "mu_a [mm^-1]", [2], "H2")
    grafico("Scattering ridotto di background (Jacques)", "mu_s' [mm^-1]", [3], "H19")
    grafico("Variazione di assorbimento dell'inclusione", "dmu_a [mm^-1]",
            list(range(4, 4 + n_var)), "H36")

<a id="prep"></a>
## 11. Preparazione per lunghezza d'onda e calibrazione sul rumore

Molte grandezze non dipendono dal singolo caso ma solo dalla lunghezza d'onda: si calcolano una volta e si riusano.

**`prepara_lambda`** calcola `W` e `frac`, i conteggi attesi $\lambda_0$ e $\sigma_0=\sqrt{2/(\lambda_0+1)}$. In modalità `"baseline"` chiama subito `prepara_pesi`.

**`prepara_pesi(sig)`** prepara tutto ciò che serve ai metodi:

| Variabile | Formula | Usata da |
|---|---|---|
| `W_w` | $W/\sigma_m$ (riga per riga, `sig[:, None]`) | tutti |
| `Uw, sw, Vtw` | SVD di $W_w$ | TSVD, Tikhonov |
| `col_L1` | $n_j=\lVert\mathbf w_j\rVert$, `np.linalg.norm(W_w, axis=0)` | L1, soglie |
| `oss_L1` | $n_j\Delta\mu_\text{max}\ge1$ (maschera booleana) | L1 |
| `B_L1` | $W_w[:,\mathcal O]\operatorname{diag}(1/n_j)$ | L1 |
| `Lip_L1` | $\lVert B\rVert_2^2$ (metodo delle potenze) | L1 |
| `peso_tv`, `peso_spigoli_tv` | $p_j$, $q_e=\sqrt{p_ip_k}$ | L1+TV |
| `rho_u_tv`, `rho_z_tv` | $c\,p_j^2$, $c\,q_e^2$ | L1+TV |
| `Minv_tv` | $\big(W_w^TW_w+P_u+D^TP_zD\big)^{-1}$ | L1+TV |

Sintassi da notare:

- `global ...` dichiara che la funzione modifica variabili definite fuori. È uno stato condiviso semplice: le funzioni di ricostruzione leggono queste variabili senza riceverle come argomenti. È comodo in uno script, ma impone di chiamare `prepara_pesi` prima di ricostruire.
- `W_w[:, oss_L1] / col_L1[oss_L1]` seleziona le colonne con una maschera booleana e divide ogni colonna per la sua norma (broadcasting sull'ultimo asse).
- `D_tv.T @ sparse.diags(rho_z_tv) @ D_tv` è un prodotto fra matrici sparse; `.toarray()` lo converte in densa per sommarlo alle altre matrici prima dell'inversione.

**`pesi_misura`** restituisce la $\sigma$ da usare: in modalità `"dati"` rifà tutta la preparazione con la $\sigma$ della misura corrente (più realistico, molto più lento).

**`ricostruisci_tutto`** chiama tutti i metodi e restituisce le ricostruzioni L1+TV in un dizionario `{nome_metodo: risultato}`.

**`calibra_soglie`** applica la procedura del §9.2: un generatore separato (`seme_rumore + 1000`) per non alterare il rumore dello scan, misure con `M_ideal = 0`, quantile con `np.quantile`, e i minimi sulle soglie L1 e L1+TV.

In [ ]:
# %% =============================================================
# PREPARAZIONE PER LAMBDA, PESI e CALIBRAZIONE DELLE SOGLIE SUL RUMORE
# ==================================================================
def prepara_lambda(mu_a0, mu_s0):
    """W e conteggi attesi di baseline per una lambda. Con modello_sigma =
    'baseline' prepara subito anche i pesi (validi per tutti i casi)."""
    global W, lam0, sig0
    W, frac_gates = matrice_W(mu_a0, mu_s0)
    lam0 = N_tot_TPSF * frac_gates                       # conteggi attesi di baseline
    sig0 = np.sqrt(2 / (lam0.flatten() + 1))             # sigma dai conteggi attesi
    if modello_sigma == "baseline":
        prepara_pesi(sig0)


def prepara_pesi(sig):
    """W_w = W/sigma, sua SVD e matrice normalizzata della L1 (variabili
    globali usate dalle funzioni di ricostruzione e dalle metriche)."""
    global W_w, Uw, sw, Vtw, col_L1, oss_L1, B_L1, Lip_L1
    global peso_tv, peso_spigoli_tv, rho_u_tv, rho_z_tv, Minv_tv
    W_w = W / sig[:, None]
    Uw, sw, Vtw = np.linalg.svd(W_w, full_matrices=False)
    col_L1 = np.linalg.norm(W_w, axis=0)
    oss_L1 = col_L1 * dmu_a_max >= 1                     # voxel osservabili
    B_L1 = W_w[:, oss_L1] / col_L1[oss_L1]
    Lip_L1 = norma2_quadra(B_L1)
    peso_tv = np.maximum(col_L1, 1 / dmu_a_max)          # pesi L1 della L1+TV (voxel)
    peso_spigoli_tv = np.sqrt(peso_tv[spigolo_piu] * peso_tv[spigolo_meno])   # pesi TV (spigoli)
    rho_u_tv = rho_rel_tv * peso_tv**2
    rho_z_tv = rho_rel_tv * peso_spigoli_tv**2
    Minv_tv = np.linalg.inv(W_w.T @ W_w + np.diag(rho_u_tv)
                            + (D_tv.T @ sparse.diags(rho_z_tv) @ D_tv).toarray())


def pesi_misura(sig_hat):
    """Sigma con cui pesare questa misura (vettore di lunghezza N_meas). Con
    modello_sigma = 'dati' ricalcola W_w, SVD e preparazione della L1."""
    if modello_sigma == "dati":
        prepara_pesi(sig_hat)
        return sig_hat
    return sig0


def ricostruisci_tutto(m_w):
    c = Uw.T @ m_w
    return (tsvd_disc(c, m_w), tikhonov_disc(c, m_w), ricostruzione_L1(m_w),
            {nome: ricostruzione_L1TV(m_w, a) for nome, a in METODI_TV.items()})


def calibra_soglie():
    """Segnale massimo 'trovato' da ciascun metodo su misure di SOLO rumore
    (dmu_a = 0): la soglia e' il suo quantile_falsi_positivi."""
    rng_cal = np.random.default_rng(seme_rumore + 1000)
    pid0 = posizioni[0][0]
    segnali = {m: [] for m in soglia_trovata}
    for wl_c in lambda_calibrazione:
        i = int(np.argmin(np.abs(wl_nm - wl_c)))
        prepara_lambda(float(mu_a_bkg[i]), float(mu_s_bkg[i]))
        for _ in range(n_realizzazioni_rumore):
            M0, sig_hat = simula_misura(lam0, np.zeros_like(lam0), rng_cal)
            sig = pesi_misura(sig_hat.flatten())
            m_w = M0.flatten() / sig
            (A1, *_), (A2, *_), (A3, *_), ris_tv = ricostruisci_tutto(m_w)
            for metodo, A_hat, f_met in ((("TSVD", A1, metriche_cluster),
                                          ("Tikhonov", A2, metriche_cluster),
                                          ("L1", A3, metriche_L1))
                                         + tuple((nome, r[0], metriche_L1)
                                                 for nome, r in ris_tv.items())):
                segnali[metodo].append(f_met(A_hat, m_w, maschere[pid0], np.zeros(3), 0.0)
                                       ["segnale_sigma"])
    # Sul solo rumore la L1+TV resta quasi sempre a zero (segnale 0 sigma), ma con
    # un segnale debole scende a lam piu' piccoli e puo' accendere componenti
    # spurie: la sua soglia non scende sotto quella della L1, che usa la stessa
    # scelta della componente. Nessuna soglia L1 scende sotto soglia_sigma_voxel.
    soglie = {m: float(np.quantile(v, quantile_falsi_positivi)) for m, v in segnali.items()}
    soglie["L1"] = max(soglie["L1"], soglia_sigma_voxel)
    for nome in METODI_TV:
        soglie[nome] = max(soglie[nome], soglie["L1"])
    return soglie, segnali

Esecuzione della calibrazione e stampa delle soglie di ogni metodo, con mediana e massimo del segnale da solo rumore.

In [ ]:
t_cal = time.time()
soglia_trovata, segnali_nulli = calibra_soglie()
print("="*78)
print(f"CALIBRAZIONE SULLE MISURE DI SOLO RUMORE ({len(lambda_calibrazione)} lambda x "
      f"{n_realizzazioni_rumore} realizzazioni, modello_sigma = '{modello_sigma}', "
      f"{time.time() - t_cal:.0f} s)")
for m, s in soglia_trovata.items():
    v = np.array(segnali_nulli[m])
    print(f"   {m:11s} segnale da solo rumore: mediana {np.median(v):4.1f}, max {v.max():4.1f} sigma "
          f"-> soglia 'trovata' = {s:.2f} sigma (quantile {quantile_falsi_positivi})")
print("="*78 + "\n")

<a id="ciclo"></a>
## 12. Ciclo principale dello scan

Per ogni lunghezza d'onda (ciclo esterno) si prepara `W`; per ogni posizione e variazione (cicli interni):

1. **misura simulata**: $\mathbf A=\delta\mu_a(\lambda)\,\mathbf 1_\text{incl}$, `M_ideal = W @ A`, poi `simula_misura` aggiunge il rumore di Poisson;
2. **sbiancamento**: `m_w = M_noisy.flatten() / sig` e coefficienti `c = Uw.T @ m_w`;
3. **SNR atteso**: la norma del segnale ideale in unità di rumore teorico,
$$
\text{SNR}=\Big\|\frac{\mathbf M_\text{ideale}}{\boldsymbol\sigma_\text{teo}}\Big\|,\qquad \sigma_\text{teo}=\sqrt{\frac{(1+M)(2+M)}{\lambda_0+1}} ,
$$
che dice quanto l'inclusione emerge dal rumore indipendentemente dal metodo (sotto circa 5 è indistinguibile);
4. **ricostruzioni** con tutti i metodi;
5. **metriche** e una riga di risultati per metodo.

### Implementazione

- `base = dict(...)` contiene le informazioni comuni del caso; `riga = dict(base, metodo=..., **met)` crea un nuovo dizionario che unisce `base`, i parametri del metodo e le metriche (`**met` "spacchetta" il dizionario delle metriche).
- `risultati_caso` è una lista di tuple `(metodo, nome_parametro, valore, chi2, metriche)`; i metodi L1+TV sono aggiunti con una *list comprehension* sul dizionario `ris_tv`.
- `METODI_TV.get(metodo, np.nan)` restituisce $\alpha$ per i metodi L1+TV e `NaN` per gli altri.
- Dopo ogni lunghezza d'onda `salva_excel` riscrive il file: se lo scan si interrompe, i risultati fin lì sono salvi. La stima del tempo rimanente è il tempo medio per caso moltiplicato per i casi mancanti.

In [ ]:
# %% =============================================================
# CICLO: lambda (esterno) x posizione x variazione (interno)
# ==================================================================
rng = np.random.default_rng(seme_rumore)
righe = []
t_inizio = time.time()
n_fatti = 0

for i_wl, wl in enumerate(wl_nm):
    mu_a0, mu_s0 = float(mu_a_bkg[i_wl]), float(mu_s_bkg[i_wl])

    # --- una volta per lambda: W e conteggi attesi (e pesi, se 'baseline') --
    prepara_lambda(mu_a0, mu_s0)

    print("#"*78)
    print(f"# LAMBDA = {wl:.0f} nm ({i_wl+1}/{N_wl}): mu_a0 = {mu_a0:.4f} mm^-1, "
          f"mu_s' = {mu_s0:.3f} mm^-1, conteggi di baseline per gate "
          f"{lam0.min():.0f}-{lam0.max():.0f}"
          + (f", {int(oss_L1.sum())} voxel osservabili L1"
             if modello_sigma == "baseline" else ""))
    print("#"*78)

    for pid, pdesc, r_pos in posizioni:
        mask = maschere[pid]
        r_true = np.array(r_pos)
        for vid, vdesc, d_lip, d_HHb, d_O2Hb in variazioni:
            dmu = float(dmu_var[vid][i_wl])
            A = mask * dmu
            M_ideal = (W @ A).reshape(N_pairs, N_gates)
            M_noisy, sig_hat = simula_misura(lam0, M_ideal, rng)
            sig = pesi_misura(sig_hat.flatten())         # 'dati': W_w e SVD rifatte qui
            m_w = M_noisy.flatten() / sig
            c = Uw.T @ m_w

            # SNR atteso: norma del segnale ideale divisa per la sigma TEORICA
            # (conteggi attesi, include l'effetto dell'inclusione sul rumore)
            sig_teo = np.sqrt((1 + M_ideal) * (2 + M_ideal) / (lam0 + 1)).flatten()
            SNR = float(np.linalg.norm(M_ideal.flatten() / sig_teo))

            A_tsvd, k_disc, chi2_tsvd = tsvd_disc(c, m_w)
            A_tik, lam_tik, chi2_tik = tikhonov_disc(c, m_w)
            A_L1, f_L1, chi2_L1 = ricostruzione_L1(m_w)
            ris_tv = {nome: ricostruzione_L1TV(m_w, a) for nome, a in METODI_TV.items()}

            base = dict(lambda_nm=wl, pos_id=pid, pos_desc=pdesc,
                        x_vero=r_pos[0], y_vero=r_pos[1], z_vero=r_pos[2],
                        var_id=vid, var_desc=vdesc, d_lip=d_lip, d_HHb=d_HHb, d_O2Hb=d_O2Hb,
                        d_tHb=d_HHb + d_O2Hb, SO2_incl=comp_var[vid]["SO2"],
                        mu_a0=mu_a0, mu_s0=mu_s0, dmu_a_vero=dmu,
                        dmu_su_mua0=dmu / mu_a0, SNR_atteso=SNR,
                        classe_SNR=classe_snr(SNR),
                        sigma_dati_su_sigma0=float(np.median(sig_hat.flatten() / sig0)),
                        voxel_osservabili_L1=int(oss_L1.sum()))
            risultati_caso = [
                ("TSVD", "k", k_disc, chi2_tsvd,
                 metriche_cluster(A_tsvd, m_w, mask, r_true, dmu, soglia_trovata["TSVD"])),
                ("Tikhonov", "lam/s0", lam_tik / sw[0], chi2_tik,
                 metriche_cluster(A_tik, m_w, mask, r_true, dmu, soglia_trovata["Tikhonov"])),
                ("L1", "lam/lam_max", f_L1, chi2_L1,
                 metriche_L1(A_L1, m_w, mask, r_true, dmu, soglia_trovata["L1"])),
            ] + [
                (nome, "lam/lam_max", f_TV, chi2_TV,
                 metriche_L1(A_TV, m_w, mask, r_true, dmu, soglia_trovata[nome]))
                for nome, (A_TV, f_TV, chi2_TV) in ris_tv.items()
            ]
            for metodo, nome_par, par, chi2_fin, met in risultati_caso:
                riga = dict(base, metodo=metodo, alpha_tv=METODI_TV.get(metodo, np.nan),
                            parametro_reg=nome_par, valore_reg=par,
                            chi2_finale=chi2_fin, chi2_atteso=N_meas, **met)
                riga["rapporto_fit"] = met["dmu_fit"] / dmu if dmu != 0 else np.nan
                riga["rapporto_fit_grezzo"] = met["dmu_fit_grezzo"] / dmu if dmu != 0 else np.nan
                riga["rapporto_picco"] = met["dmu_picco"] / dmu if dmu != 0 else np.nan
                riga["segno_corretto"] = bool(np.sign(met["dmu_fit"]) == np.sign(dmu)) \
                    if np.isfinite(met["dmu_fit"]) else False
                righe.append(riga)

            # --- stampa compatta del confronto ---------------------------------
            n_fatti += 1
            print(f"{wl:.0f} nm | {pid} {r_pos} | {vid} {vdesc:38s} | dmu_a = {dmu:+.2e} "
                  f"({100*dmu/mu_a0:+.0f}% di mu_a0) | SNR atteso {SNR:7.1f}")
            for metodo, nome_par, par, chi2_fin, met in risultati_caso:
                extra = f", {met['n_componenti']} comp." if metodo.startswith("L1") else ""
                if met["trovata"]:
                    esito_txt = (f"errore {met['errore_baricentro_mm']:5.1f} mm  dmu fit/vero "
                                 f"{met['dmu_fit']/dmu if dmu else np.nan:7.2f}")
                else:
                    esito_txt = "NON TROVATA                         "
                print(f"     {metodo:11s} {nome_par}={par:<9.3g}  {esito_txt}"
                      f"  segnale {met['segnale_sigma']:7.1f} sigma"
                      f"  voxel {met['voxel_ricostruiti']:3d} (in comune {met['voxel_in_comune']}/8)"
                      f"{extra}")

    # --- salvataggio dopo ogni lambda (lo scan resta salvato se si interrompe)
    salva_excel(righe, file_risultati)
    trascorso = time.time() - t_inizio
    stima = trascorso / n_fatti * (N_casi - n_fatti)
    print(f"\n>>> {n_fatti}/{N_casi} casi, {trascorso/60:.1f} min trascorsi, "
          f"~{stima/60:.0f} min rimanenti. Salvato in '{file_risultati}'.\n")

<a id="riepilogo"></a>
## 13. Riepilogo

Le tabelle finali sono `pivot_table` sul `DataFrame` dei risultati:

- errore mediano del baricentro e voxel in comune, per posizione e metodo;
- rapporto mediano fra $\delta\mu_a$ stimato e vero, per variazione e metodo;
- frazione di inclusioni trovate e numero di casi per classe di SNR.

Le mediane su errore e rapporto sono calcolate solo sui casi con SNR atteso $\ge20$: sotto, l'inclusione si confonde con il rumore e i rapporti non hanno significato. Tutti i casi restano comunque nel file Excel, filtrabili con `classe_SNR`.

In [ ]:
# %% =============================================================
# STEP 12 - Riepilogo finale
# ==================================================================
df = salva_excel(righe, file_risultati)
print("="*78)
print(f"SCAN COMPLETATO in {(time.time() - t_inizio)/60:.1f} min "
      f"(modello_sigma = '{modello_sigma}'). Risultati in '{file_risultati}'")
print("="*78)
pd.set_option("display.width", 160)
# mediane solo sui casi con SNR atteso >= 20: sotto, l'inclusione e' confusa col
# rumore e i rapporti stimato/vero non hanno significato (tutti i casi sono
# comunque nel file Excel, filtrabili con la colonna classe_SNR)
ril = df[df["SNR_atteso"] >= 20]
print(f"\nCasi con SNR atteso >= 20: {ril.shape[0] // len(METODI)} su {df.shape[0] // len(METODI)}")
print("\nErrore mediano sul baricentro [mm] (posizione x metodo, SNR atteso >= 20):")
print(ril.pivot_table(index="pos_id", columns="metodo", values="errore_baricentro_mm",
                      aggfunc="median").round(1))
print("\nVoxel ricostruiti in comune con l'inclusione (mediana su 8, SNR atteso >= 20):")
print(ril.pivot_table(index="pos_id", columns="metodo", values="voxel_in_comune",
                      aggfunc="median").round(1))
print("\nRapporto mediano dmu_a fit / vero (variazione x metodo, SNR atteso >= 20):")
print(ril.pivot_table(index="var_id", columns="metodo", values="rapporto_fit",
                      aggfunc="median").round(2))
print("\nFrazione di casi con inclusione TROVATA (posizione x metodo, tutti i casi):")
print(df.pivot_table(index="pos_id", columns="metodo", values="trovata",
                     aggfunc="mean").round(2))
print("\nNumero di casi per classe di SNR atteso:")
print((df.groupby("classe_SNR").size() // len(METODI)).to_string())
print(f"\nRapporto mediano sigma stimata dai dati / sigma_0 di baseline: "
      f"{df['sigma_dati_su_sigma0'].median():.3f}")

<a id="finali"></a>
## 14. Considerazioni finali: verso una ricostruzione più realistica

Lo scan, così com'è, misura le prestazioni dei metodi in condizioni **ideali**: i dati sono generati con lo stesso modello lineare usato per ricostruire (*inverse crime*), il background è omogeneo e semi-infinito, il rumore è solo Poisson con varianza nota. Anche restando in simulazione, i passi successivi per avvicinarsi al caso reale sono:

1. **Separare il modello dei dati da quello della ricostruzione.** Generare le misure con un modello più fedele (Monte Carlo, ad esempio MCX/`pmcx` su GPU, che lavora sulla stessa griglia di voxel; oppure la perturbazione non lineare dell'inclusione finita) e ricostruire con la `W` diffusiva. Misura l'effetto dell'errore di modello, che con dati reali c'è sempre. Il Monte Carlo permette anche di verificare `W` stessa: $-W_{(p,g),j}$ è il cammino medio dei fotoni nel voxel, che il Monte Carlo calcola esattamente; le zone critiche sono il primo gate e i voxel superficiali vicini alle fibre (rassegna §5: la diffusione sbaglia per $\mu_a>0.01$ mm⁻¹ entro 5 mm dalla sorgente).
2. **Includere l'IRF** del sistema (circa 120 ps FWHM nel vostro caso) nel modello diretto, convolvendo $\Phi_0$ e $\delta\Phi$, e poi nell'inversione.
3. **Rumore e scelta di $\lambda$ realistici.** Stimare $\sigma$ da ripetizioni della baseline invece che dal solo modello di Poisson; aggiungere un fattore $\tau>1$ alla discrepanza ($\chi^2=\tau N_\text{meas}$) e confrontarla con L-curve, GCV o cross-validazione sulle coppie sorgente-rivelatore, che non richiedono di conoscere $\sigma$ (rassegna §4.4).
4. **Background non omogeneo.** Strati (pelle, grasso, muscolo) o una geometria reale, eventualmente con informazioni strutturali da ecografia (hard o soft prior, rassegna §4.2; pesi della TV ridotti sui bordi noti).
5. **Oltre Born.** Per variazioni di decine di percento di $\mu_{a0}$ la linearizzazione non basta: ricostruzione non lineare iterativa (Gauss–Newton, rassegna §2.3), ricalcolando `W` attorno alla stima corrente.
6. **Ricostruzione spettrale diretta.** Ricostruire $\delta C_\text{HHb}$, $\delta C_\text{O_2Hb}$ e $\delta f_\text{lip}$ usando tutte le 31 lunghezze d'onda insieme (rassegna §4.3), invece di $\delta\mu_a$ lunghezza d'onda per lunghezza d'onda. Riduce le incognite, media il rumore, impone che l'inclusione sia nello stesso posto a tutte le $\lambda$, e permette di misurare il cross-talk fra cromofori. È anche la naturale estensione della L1 e della L1+TV: i cromofori dello stesso voxel formano un gruppo (*group sparsity*).
7. **Ricostruire anche $\mu_s'$** (con il suo prior spettrale $a\lambda^{-b}$): con dati reali non è noto, e il dominio del tempo permette di separarlo da $\mu_a$.
8. **Inclusioni più varie.** Dimensioni diverse da 2×2×2 (per verificare che i risultati non dipendano dall'aver indovinato la taglia, e per scegliere $\alpha$ in modo robusto) e inclusioni eterogenee, ad esempio bordo ad alta emoglobina e nucleo ipossico (rassegna §5).

<a id="riferimenti"></a>
## 15. Riferimenti

**Rassegna di riferimento**
- S. Okawa, Y. Hoshi, *A Review of Image Reconstruction Algorithms for Diffuse Optical Tomography*, Appl. Sci. 13, 5016 (2023). https://doi.org/10.3390/app13085016

**Modello diretto e ottica dei tessuti**
- S. R. Arridge, *Optical tomography in medical imaging*, Inverse Probl. 15, R41 (1999).
- F. Martelli, S. Del Bianco, A. Ismaelli, G. Zaccanti, *Light Propagation through Biological Tissue and Other Diffusive Media*, SPIE Press (2010).
- S. L. Jacques, *Optical properties of biological tissues: a review*, Phys. Med. Biol. 58, R37 (2013).

**L1, soglia morbida e FISTA**
- R. Tibshirani, *Regression shrinkage and selection via the lasso*, J. R. Stat. Soc. B 58, 267 (1996).
- I. Daubechies, M. Defrise, C. De Mol, *An iterative thresholding algorithm for linear inverse problems with a sparsity constraint*, Comm. Pure Appl. Math. 57, 1413 (2004).
- A. Beck, M. Teboulle, *A fast iterative shrinkage-thresholding algorithm for linear inverse problems*, SIAM J. Imaging Sci. 2, 183 (2009).
- N. Parikh, S. Boyd, *Proximal Algorithms*, Found. Trends Optim. 1(3) (2014).
- E. T. Hale, W. Yin, Y. Zhang, *Fixed-point continuation for ℓ1-minimization*, SIAM J. Optim. 19, 1107 (2008).
- D. L. Donoho, I. M. Johnstone, *Ideal spatial adaptation by wavelet shrinkage*, Biometrika 81, 425 (1994).

**L1 + variazione totale e ADMM**
- R. Tibshirani, M. Saunders, S. Rosset, J. Zhu, K. Knight, *Sparsity and smoothness via the fused lasso*, J. R. Stat. Soc. B 67, 91 (2005).
- J. Dutta, S. Ahn, C. Li, S. R. Cherry, R. M. Leahy, *Joint L1 and total variation regularization for fluorescence molecular tomography*, Phys. Med. Biol. 57, 1459 (2012).
- S. Boyd, N. Parikh, E. Chu, B. Peleato, J. Eckstein, *Distributed optimization and statistical learning via the alternating direction method of multipliers*, Found. Trends Mach. Learn. 3(1) (2011).

**Scelta del parametro di regolarizzazione**
- P. C. Hansen, *Discrete Inverse Problems: Insight and Algorithms*, SIAM (2010).